
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>

#Lab - Context Is Everything - Lakebase Agent Memory

In this hands-on lab, you will build a complete stateful AI assistant for **Bakehouse**, a global bakery franchise chain with 48 locations across 9 countries. You will implement short-term memory, long-term memory, and production deployment on Databricks.


**The Scenario**


Bakehouse franchise managers need an AI assistant that can:


- **Analyze sales performance**: Find top-selling products, compare revenue across locations, and identify trends by franchise or region. *Powered by **UC Function Tools** that query live bakehouse data.*
- **Compare franchise locations**: Benchmark one franchise against others in the same country or city, including average order value and transaction volume. *Powered by **UC Function Tools** with flexible filtering by country, city, or franchise name.*
- **Maintain conversation context**: Follow multi-turn conversations where "those franchises" or "our top product" references earlier answers without repetition. *Powered by **Short-Term Memory** (CheckpointSaver) that persists the full conversation within a session thread.*
- **Remember each manager across sessions**: Recall the manager's name, franchise, preferred products, and reporting preferences, even days later in a completely new browser session. *Powered by **Long-Term Memory** (DatabricksStore) that saves curated facts per user, scoped by namespace.*
- **Run in production**: Deploy as a Databricks App with a chat interface that franchise managers can use from any browser. *Both memory systems use their **async counterparts** (AsyncCheckpointSaver, AsyncDatabricksStore) which are required for non-blocking operation in the async FastAPI server.*

**What You Will Build**


You will build this assistant step by step in this lab:


1. **UC Function Tools**: SQL functions that query `samples.bakehouse` sales data, registered in Unity Catalog and callable by the agent
1. **Agent Memory**: CheckpointSaver for short-term conversation state, plus `save_memory` and `recall_memories` tools backed by DatabricksStore for cross-session knowledge
1. **LangGraph Agent**: A stateful agent graph with short-term memory (CheckpointSaver) for conversation continuity and long-term memory (DatabricksStore) for cross-session recall
1. **Databricks App**: A production deployment with FastAPI, async memory, and a branded chat interface



<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color:#c62828; font-size: 14pt;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2574B5;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies" style="color: #2574B5;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2574B5;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>: Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Lab Setup

### A1. Run Lab Setup and Load Configuration

Run the setup script below. It installs the required packages, creates your user-scoped Unity Catalog catalog and schema, writes the agent configuration YAML, and generates the Databricks App files you will deploy later. It also derives your Lakebase project name (you will create the instance manually in Section B2).


In [0]:
%run ./Includes/Classroom-Setup-Lakebase-Memory

### A2. Load Configuration

Load the Lakebase connection details and agent configuration from the YAML file at `./artifacts/configs/bakehouse_agent_config.yaml`. This file was written by the lab setup with your environment's specific values.


<table style="border-collapse: collapse; width: 100%; font-size: 14px;">
<tr style="border-bottom: 2px solid #ddd;">
  <th style="text-align: left; padding: 8px; width: 35%;">Key</th>
  <th style="text-align: left; padding: 8px;">Purpose</th>
</tr>
<tr style="border-bottom: 1px solid #eee;">
  <td style="padding: 8px;"><code>LLM_ENDPOINT_NAME</code></td>
  <td style="padding: 8px;">The Databricks Model Serving endpoint for LLM inference.</td>
</tr>
<tr style="border-bottom: 1px solid #eee;">
  <td style="padding: 8px;"><code>CATALOG_NAME</code> / <code>SCHEMA_NAME</code></td>
  <td style="padding: 8px;">Your user-scoped Unity Catalog location where the agent's tools are registered.</td>
</tr>
<tr style="border-bottom: 1px solid #eee;">
  <td style="padding: 8px;"><code>LAKEBASE_AUTOSCALING_PROJECT</code> / <code>LAKEBASE_AUTOSCALING_BRANCH</code></td>
  <td style="padding: 8px;">Identifies the dedicated Lakebase instance for both checkpoint storage (short-term) and the memory store (long-term).</td>
</tr>
<tr style="border-bottom: 1px solid #eee;">
  <td style="padding: 8px;"><code>TOOL1</code> / <code>TOOL2</code></td>
  <td style="padding: 8px;">The two Unity Catalog functions (<code>top_products_by_franchise</code> and <code>franchise_performance</code>) that query bakehouse sales data. These will be created and stored in schema <code>bakehouse</code>.</td>
</tr>
<tr>
  <td style="padding: 8px;"><code>EMBEDDING_ENDPOINT</code> / <code>EMBEDDING_DIMS</code></td>
  <td style="padding: 8px;">The embedding model and dimension size used by <code>DatabricksStore</code> for semantic search. This enables <code>recall_memories</code> to find relevant facts by meaning, not just exact key match.</td>
</tr>
</table>

In [0]:
import mlflow

# Load the agent configuration written by the lab setup
# This YAML file contains catalog, schema, Lakebase, and LLM settings
agent_config_file = "./artifacts/configs/bakehouse_agent_config.yaml"
model_config = mlflow.models.ModelConfig(development_config=agent_config_file)

# Extract configuration values used throughout the notebook
catalog_name = model_config.get("CATALOG_NAME")
schema_name = model_config.get("SCHEMA_NAME")
lakebase_autoscaling_project = model_config.get("LAKEBASE_AUTOSCALING_PROJECT")
lakebase_autoscaling_branch = model_config.get("LAKEBASE_AUTOSCALING_BRANCH")
LLM_ENDPOINT_NAME = model_config.get("LLM_ENDPOINT_NAME")
EMBEDDING_ENDPOINT = model_config.get("EMBEDDING_ENDPOINT")
EMBEDDING_DIMS = model_config.get("EMBEDDING_DIMS")
TOOL_1 = model_config.get("TOOL1")
TOOL_2 = model_config.get("TOOL2")
app_name = model_config.get("APP_NAME")
SYSTEM_PROMPT = model_config.get("SYSTEM_PROMPT")

# Configure MLflow to log all agent traces to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.langchain.autolog()  # Automatically trace all LangChain/LangGraph calls
mlflow.set_experiment(experiment_location)  # experiment_location was set by the lab setup

print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"LLM:      {LLM_ENDPOINT_NAME}")
print(f"Lakebase: {lakebase_autoscaling_project} / {lakebase_autoscaling_branch}")

### A3. Explore the Bakehouse Data

Before diving in, let's understand the data your agent will work with. The `samples.bakehouse` schema contains sales data for a global bakery franchise chain.


In [0]:
# View the franchise locations
display(spark.sql("""
    SELECT name AS `Franchise Name`,
           city AS `City`,
           district AS `District`,
           country AS `Country`,
           size AS `Size`
    FROM samples.bakehouse.sales_franchises
    ORDER BY country, city
"""))

In [0]:
# View available products and total sales
display(spark.sql("""
    SELECT product AS `Product`,
           COUNT(*) AS `Transactions`,
           SUM(CAST(quantity AS INT)) AS `Total Quantity`,
           ROUND(SUM(CAST(totalPrice AS DOUBLE)), 2) AS `Total Revenue ($)`
    FROM samples.bakehouse.sales_transactions
    GROUP BY product
    ORDER BY `Total Revenue ($)` DESC
"""))

In [0]:
# Verify the date range of the data
display(spark.sql("""
    SELECT DATE_FORMAT(MIN(dateTime), 'MMMM d, yyyy') AS `Earliest Transaction`,
           DATE_FORMAT(MAX(dateTime), 'MMMM d, yyyy') AS `Latest Transaction`,
           FORMAT_NUMBER(COUNT(*), 0) AS `Total Transactions`
    FROM samples.bakehouse.sales_transactions
"""))


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul style="margin-bottom: 0;">
    <li><strong>Bakehouse has 48 franchises across 9 countries</strong>, with the largest presence in Japan (20) and the US (16).</li>
    <li>There are <strong>6 products</strong>: each named after a city or region, reflecting the franchise's global identity.</li>
    <li>Your UC function tools will query this data to answer franchise managers' questions.</li>
  </ul>
</div>

### A4. The Four Context Sources

Every agent call assembles **multiple context sources** into a single prompt, organized into three categories: **Static** (set once at design time), **Stateful** (memory that persists across interactions), and **Retrieval** (data fetched on demand). The quality of the agent's response depends heavily on the quality of these inputs.


<!-- ── Visual: context flow into agent ── -->
<style>
.code-block-dark { background: #2d2d2d !important; border-radius: 8px; padding: 14px 18px; font-family: Consolas, Monaco, 'Andale Mono', monospace; font-size: 14pt; color: #ccc; line-height: 1.6; overflow-x: auto; margin: 8px 0; }
.code-block-dark .kw { color: #cc99cd; }
.code-block-dark .fn { color: #f08d49; }
.code-block-dark .st { color: #7ec699; }
.code-block-dark .cm { color: #999; font-style: italic; }
.code-block-dark .op { color: #67cdcc; }
.code-block-dark .num { color: #f8c555; }
.code-block-dark .cls { color: #f8c555; }
.code-block-dark .dc { color: #f8c555; }
/* ═══ Standard Code Block ═══ */
.a3f { max-width: 1200px; margin: 20px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; }
/* Card row */
.a3f-row { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; }
.a3f-card { background: #fff; border-radius: 10px; overflow: hidden; box-shadow: 0 2px 8px rgba(27,49,57,0.10), 0 1px 3px rgba(27,49,57,0.06); display: flex; flex-direction: column; }
.a3f-accent { height: 7px; flex-shrink: 0; }
.a3f-accent.blue { background: #1B3139; }
.a3f-accent.teal { background: #1B5162; }
.a3f-accent.green { background: #618794; }
.a3f-accent.amber { background: #90A5B1; }
.a3f-body { padding: 14px 13px 16px; flex: 1; }
.a3f-pill { display: inline-block; font-size: 14pt; font-weight: 700; letter-spacing: 0.8px; text-transform: uppercase; padding: 2px 9px; border-radius: 20px; margin-bottom: 8px; }
.a3f-pill.static { background: #EEEDE9; color: #1B5162; }
.a3f-pill.dynamic { background: #EEEDE9; color: #5A6F77; }
.a3f-ctitle { font-size: 14pt; font-weight: 700; color: #1B3139; margin-bottom: 8px; }
.a3f-body ul { list-style: none; padding: 0; margin: 0; }
.a3f-body li { font-size: 14pt; color: #3d5a65; line-height: 1.45; padding: 2px 0 2px 13px; position: relative; }
.a3f-body li::before { content: ''; position: absolute; left: 0; top: 9px; width: 5px; height: 5px; border-radius: 50%; background: #c4d0d5; }
/* Connector zone */
.a3f-conn { position: relative; height: 70px; }
.a3f-vlines { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; height: 20px; }
.a3f-vl { display: flex; justify-content: center; }
.a3f-vl::after { content: ''; display: block; width: 2px; height: 100%; background: #94b3be; }
.a3f-hbar { position: absolute; top: 19px; left: 12%; right: 12%; height: 3px; background: #94b3be; border-radius: 2px; }
.a3f-funnel { position: absolute; top: 22px; left: 50%; transform: translateX(-50%); width: 0; height: 0; border-left: 70px solid transparent; border-right: 70px solid transparent; border-top: 20px solid rgba(148,179,190,0.15); }
.a3f-cdrop { position: absolute; top: 19px; left: 50%; transform: translateX(-50%); width: 3px; height: 34px; background: #94b3be; border-radius: 2px; }
.a3f-cdrop::after { content: ''; position: absolute; bottom: -7px; left: 50%; transform: translateX(-50%); width: 0; height: 0; border-left: 7px solid transparent; border-right: 7px solid transparent; border-top: 8px solid #1B5162; }
/* Agent */
.a3f-agent { background: #1B5162; color: #fff; max-width: 360px; margin: 0 auto; border-radius: 14px; padding: 18px 24px; text-align: center; box-shadow: 0 4px 16px rgba(27,49,57,0.22); }
.a3f-agent-t { font-size: 16pt; font-weight: 700; margin-bottom: 4px; }
.a3f-agent-s { font-size: 14pt; color: #a8c9d6; }
/* Output */
.a3f-outconn { display: flex; flex-direction: column; align-items: center; height: 40px; }
.a3f-outconn::before { content: ''; display: block; width: 2px; height: 28px; background: #b0c4cc; }
.a3f-outconn::after { content: ''; display: block; width: 0; height: 0; border-left: 6px solid transparent; border-right: 6px solid transparent; border-top: 7px solid #1B5162; }
/* Response */
.a3f-resp { background: #fff; border: 2px solid #d6e0e4; border-radius: 12px; max-width: 420px; margin: 0 auto; padding: 16px 24px; text-align: center; box-shadow: 0 2px 8px rgba(27,49,57,0.07); }
.a3f-resp-t { font-size: 14pt; font-weight: 700; color: #1B3139; margin-bottom: 3px; }
.a3f-resp-s { font-size: 14pt; color: #618794; }
</style>

<div class="a3f">
  <div class="a3f-row">
    <div class="a3f-card"><div class="a3f-accent blue"></div><div class="a3f-body"><span class="a3f-pill static">Static</span><div class="a3f-ctitle">System Prompt</div><ul><li>Role and persona definition</li><li>Behavioral guardrails</li><li>Output format instructions</li><li>Tool-use permissions</li></ul></div></div>
    <div class="a3f-card"><div class="a3f-accent teal"></div><div class="a3f-body"><span class="a3f-pill static">Stateful</span><div class="a3f-ctitle">Short-Term Memory</div><ul><li>Current conversation turns</li><li>Recent tool call results</li><li>Working scratchpad state</li></ul></div></div>
    <div class="a3f-card"><div class="a3f-accent green"></div><div class="a3f-body"><span class="a3f-pill static">Stateful</span><div class="a3f-ctitle">Long-Term Memory</div><ul><li>Persisted user preferences</li><li>Cross-session summaries</li><li>Learned patterns &amp; facts</li></ul></div></div>
    <div class="a3f-card"><div class="a3f-accent amber"></div><div class="a3f-body"><span class="a3f-pill dynamic">Retrieval</span><div class="a3f-ctitle">Retrieved Data</div><ul><li>RAG document chunks</li><li>API &amp; database results</li><li>Web search snippets</li></ul></div></div>
  </div>
  <div class="a3f-conn"><div class="a3f-vlines"><div class="a3f-vl"></div><div class="a3f-vl"></div><div class="a3f-vl"></div><div class="a3f-vl"></div></div><div class="a3f-hbar"></div><div class="a3f-funnel"></div><div class="a3f-cdrop"></div></div>
  <div class="a3f-agent"><div class="a3f-agent-t">LLM Agent</div><div class="a3f-agent-s">Assembles all context into a single prompt</div></div>
  <div class="a3f-outconn"></div>
  <div class="a3f-resp"><div class="a3f-resp-t">Contextual Response</div><div class="a3f-resp-s">Informed by all four sources</div></div>
</div>
<br/>
<style>
details summary span:first-child {
transition: transform 0.2s ease;
display: inline-block;
}
details[open] summary span:first-child {
transform: rotate(90deg);
}
</style>


## B. Build the Agent

In this section, you will create every component of the Bakehouse Sales Assistant and wire them together.


<div style="max-width: 720px; margin: 24px auto; font-family: -apple-system, sans-serif; font-size: 14pt;">

<!-- Top row: Tools -->
<div style="display: flex; gap: 14px; margin-bottom: 6px;">
  <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 8px; padding: 16px 18px; background: #f8fafc; text-align: center;">
    <div style="font-weight: 700; color: #1B3A4B; font-size: 15pt;">UC Function Tools</div>
    <div style="color: #666; font-size: 14pt; margin-top: 6px; line-height: 1.5;"><code style="font-size: 14pt;">top_products_by_franchise</code><br><code style="font-size: 14pt;">franchise_performance</code></div>
  </div>
  <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 8px; padding: 16px 18px; background: #f8fafc; text-align: center;">
    <div style="font-weight: 700; color: #1B3A4B; font-size: 15pt;">Agent Memory</div>
    <div style="color: #666; font-size: 14pt; margin-top: 6px; line-height: 1.5;"><code style="font-size: 14pt;">CheckpointSaver</code><br><code style="font-size: 14pt;">save_memory</code> / <code style="font-size: 14pt;">recall_memories</code></div>
  </div>
</div>

<!-- Arrows -->
<div style="display: flex; justify-content: space-around; padding: 0 60px;">
  <div style="width: 2px; height: 18px; background: #1B3A4B; margin: 0 auto;"></div>
  <div style="width: 2px; height: 18px; background: #1B3A4B; margin: 0 auto;"></div>
</div>

<!-- Center: Agent -->
<div style="border: 2.5px solid #1B3A4B; border-radius: 10px; padding: 18px 24px; background: #1B3A4B; color: white; text-align: center;">
  <div style="font-weight: 700; font-size: 18pt; letter-spacing: 0.3px;">Agent</div>
  <div style="font-size: 14pt; opacity: 0.8; margin-top: 5px;">System prompt &bull; ReAct loop &bull; Decides which tools to call</div>
</div>

<!-- Arrow -->
<div style="width: 2px; height: 18px; background: #8B4513; margin: 0 auto;"></div>

<!-- Bottom: Lakebase -->
<div style="border: 1.5px solid #8B4513; border-radius: 10px; overflow: hidden;">
  <div style="background: #8B4513; color: white; text-align: center; padding: 10px; font-weight: 600; font-size: 14pt; letter-spacing: 0.8px;">LAKEBASE</div>
  <div style="display: flex;">
    <div style="flex: 1; padding: 14px 18px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;">
      <div style="font-weight: 700; color: #8B4513; font-size: 15pt;">CheckpointSaver</div>
      <div style="color: #666; font-size: 14pt; margin-top: 4px;">Short-term memory<br>per thread</div>
    </div>
    <div style="flex: 1; padding: 14px 18px; text-align: center; background: #FFF8F0;">
      <div style="font-weight: 700; color: #8B4513; font-size: 15pt;">DatabricksStore</div>
      <div style="color: #666; font-size: 14pt; margin-top: 4px;">Long-term memory<br>per manager</div>
    </div>
  </div>
</div>

</div>

### B1. Create UC Function Tools

Create two Unity Catalog SQL functions that let the agent query real bakehouse sales data.

#### B1a. Create `top_products_by_franchise`

This function finds the best-selling products by franchise or country, ranked by revenue. The agent can call it when a manager asks about product performance.

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Mini architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; border-left: 3px solid #FF6B35;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 2px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 2px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <div style="display: flex; justify-content: space-around; padding: 0 30px;"><div style="width: 2px; height: 8px; background: #1B3A4B;"></div><div style="width: 2px; height: 8px; background: #1B3A4B;"></div></div>
      <div style="border: 2px solid #1B3A4B; border-radius: 6px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; opacity: 0.3;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8;">System prompt &bull; ReAct loop</div>
      </div>
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto; opacity: 0.3;"></div>
      <div style="border: 1.5px solid #8B4513; border-radius: 4px; overflow: hidden; opacity: 0.3;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 4px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;"><div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div><div style="color: #666; font-size: 8pt;">Short-term</div></div>
          <div style="flex: 1; padding: 4px 6px; text-align: center; background: #FFF8F0;"><div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div><div style="color: #666; font-size: 8pt;">Long-term</div></div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 8px 14px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">Tool 1: top_products_by_franchise</code></div>
      <div style="padding: 14px; background: #f8fafc; display: flex; gap: 10px; align-items: center;">
        <div style="flex: 1; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Input</div>
          <div style="font-size: 13pt; color: #333; margin-top: 4px;"><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">filter_franchise</code><br><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">filter_country</code></div>
        </div>
        <div style="font-size: 16pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1.2; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Joins + Aggregates</div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">sales_transactions<br>&#x2a1d; sales_franchises</div>
        </div>
        <div style="font-size: 16pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Output</div>
          <div style="font-size: 13pt; color: #333; margin-top: 4px;">Product rankings<br>by revenue</div>
        </div>
      </div>
    </div>
  </div>
</div>


In [0]:
%sql
-- Create a UC table function that returns top products for a given franchise or country
-- Parameters are optional: pass NULL to skip a filter
-- The COMMENT is important. It tells the LLM what this tool does
CREATE OR REPLACE FUNCTION top_products_by_franchise(
    filter_franchise STRING DEFAULT NULL COMMENT 'Filter by franchise name (partial match, case-insensitive). Pass NULL to include all franchises.',
    filter_country STRING DEFAULT NULL COMMENT 'Filter by country (partial match, case-insensitive). Pass NULL to include all countries.'
)
RETURNS TABLE (franchise_name STRING, product STRING, total_quantity BIGINT, total_revenue DOUBLE, num_transactions BIGINT)
COMMENT 'Find top-selling products across Bakehouse franchises. Filter by filter_franchise (franchise name) or filter_country. Returns product sales ranked by revenue.'
RETURN
SELECT
    f.name as franchise_name,
    t.product,
    SUM(CAST(t.quantity AS BIGINT)) as total_quantity,
    SUM(CAST(t.totalPrice AS DOUBLE)) as total_revenue,
    COUNT(*) as num_transactions
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_franchises f ON t.franchiseID = f.franchiseID
WHERE (filter_franchise IS NULL OR LOWER(f.name) LIKE CONCAT('%', LOWER(filter_franchise), '%'))
  AND (filter_country IS NULL OR LOWER(f.country) LIKE CONCAT('%', LOWER(filter_country), '%'))
GROUP BY f.name, t.product
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Test it: What are Golden Crumbs' top products?
SELECT * FROM top_products_by_franchise('Golden Crumbs', NULL)

<details>
<summary style="cursor: pointer; font-size: 14pt; color: #618794; font-weight: 600;">Optional exploration: try changing the parameters</summary>
<div style="margin-top: 12px;">
  <div style="position: relative; max-width: 900px; margin: 8px 0;">
    <div id="tpf1" style="background: #1b3139; border-radius: 8px; padding: 14px 18px; color: #e0e0e0; font-family: monospace; font-size: 13pt; line-height: 1.6; border-left: 4px solid #1B5162;">SELECT * FROM top_products_by_franchise(NULL, 'Japan')</div>
    <div style="font-size: 13pt; color: #618794; margin: 4px 0 12px 0;">Top products across all Japanese franchises</div>
    <button onclick="var t=document.getElementById('tpf1').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:8px;right:8px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <div style="position: relative; max-width: 900px; margin: 8px 0;">
    <div id="tpf2" style="background: #1b3139; border-radius: 8px; padding: 14px 18px; color: #e0e0e0; font-family: monospace; font-size: 13pt; line-height: 1.6; border-left: 4px solid #1B5162;">SELECT * FROM top_products_by_franchise('Tokyo', 'Japan')</div>
    <div style="font-size: 13pt; color: #618794; margin: 4px 0 12px 0;">Filter by both franchise name and country</div>
    <button onclick="var t=document.getElementById('tpf2').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:8px;right:8px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <div style="position: relative; max-width: 900px; margin: 8px 0;">
    <div id="tpf3" style="background: #1b3139; border-radius: 8px; padding: 14px 18px; color: #e0e0e0; font-family: monospace; font-size: 13pt; line-height: 1.6; border-left: 4px solid #1B5162;">SELECT * FROM top_products_by_franchise(NULL, NULL)</div>
    <div style="font-size: 13pt; color: #618794; margin: 4px 0 0 0;">Global top products across all 48 franchises</div>
    <button onclick="var t=document.getElementById('tpf3').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:8px;right:8px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
</div>
</details>

#### B1b. Create `franchise_performance`

This function compares performance across franchise locations with revenue, order count, and average order value. The agent can call it when a manager asks to compare franchises.

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Mini architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; border-left: 3px solid #FF6B35;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 2px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 2px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <div style="display: flex; justify-content: space-around; padding: 0 30px;"><div style="width: 2px; height: 8px; background: #1B3A4B;"></div><div style="width: 2px; height: 8px; background: #1B3A4B;"></div></div>
      <div style="border: 2px solid #1B3A4B; border-radius: 6px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; opacity: 0.3;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8;">System prompt &bull; ReAct loop</div>
      </div>
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto; opacity: 0.3;"></div>
      <div style="border: 1.5px solid #8B4513; border-radius: 4px; overflow: hidden; opacity: 0.3;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 4px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;"><div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div><div style="color: #666; font-size: 8pt;">Short-term</div></div>
          <div style="flex: 1; padding: 4px 6px; text-align: center; background: #FFF8F0;"><div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div><div style="color: #666; font-size: 8pt;">Long-term</div></div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 8px 14px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">Tool 2: franchise_performance</code></div>
      <div style="padding: 14px; background: #f8fafc; display: flex; gap: 10px; align-items: center;">
        <div style="flex: 1; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Input</div>
          <div style="font-size: 13pt; color: #333; margin-top: 4px;"><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">filter_country</code><br><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">filter_city</code></div>
        </div>
        <div style="font-size: 16pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1.2; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Joins + Aggregates</div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">sales_transactions<br>&#x2a1d; sales_franchises</div>
        </div>
        <div style="font-size: 16pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Output</div>
          <div style="font-size: 13pt; color: #333; margin-top: 4px;">Revenue, orders,<br>avg order value</div>
        </div>
      </div>
    </div>
  </div>
</div>


In [0]:
%sql
-- Create a UC table function that compares franchise performance
-- Joins transactions with franchise metadata to compute per-location metrics
CREATE OR REPLACE FUNCTION franchise_performance(
    filter_country STRING DEFAULT NULL COMMENT 'Filter by country (partial match, case-insensitive). Pass NULL to include all countries.',
    filter_city STRING DEFAULT NULL COMMENT 'Filter by city (partial match, case-insensitive). Pass NULL to include all cities.'
)
RETURNS TABLE (franchise_name STRING, city STRING, district STRING, country STRING, total_revenue DOUBLE, total_orders BIGINT, avg_order_value DOUBLE)
COMMENT 'Compare performance across Bakehouse franchise locations. Filter by filter_country or filter_city. Returns revenue, order count, and average order value per franchise.'
RETURN
SELECT
    f.name as franchise_name,
    f.city,
    f.district,
    f.country,
    SUM(CAST(t.totalPrice AS DOUBLE)) as total_revenue,
    COUNT(*) as total_orders,
    ROUND(AVG(CAST(t.totalPrice AS DOUBLE)), 2) as avg_order_value
FROM samples.bakehouse.sales_transactions t
JOIN samples.bakehouse.sales_franchises f ON t.franchiseID = f.franchiseID
WHERE (filter_country IS NULL OR LOWER(f.country) LIKE CONCAT('%', LOWER(filter_country), '%'))
  AND (filter_city IS NULL OR LOWER(f.city) LIKE CONCAT('%', LOWER(filter_city), '%'))
GROUP BY f.name, f.city, f.district, f.country
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Test it: Compare the two Seattle franchises
SELECT * FROM franchise_performance('US', 'Seattle')

<details>
<summary style="cursor: pointer; font-size: 14pt; color: #618794; font-weight: 600;">Optional exploration: try different filters</summary>
<div style="margin-top: 12px;">
  <div style="position: relative; max-width: 900px; margin: 8px 0;">
    <div id="fp1" style="background: #1b3139; border-radius: 8px; padding: 14px 18px; color: #e0e0e0; font-family: monospace; font-size: 13pt; line-height: 1.6; border-left: 4px solid #1B5162;">SELECT * FROM franchise_performance('Japan', NULL)</div>
    <div style="font-size: 13pt; color: #618794; margin: 4px 0 12px 0;">All Japanese franchises ranked by revenue</div>
    <button onclick="var t=document.getElementById('fp1').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:8px;right:8px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <div style="position: relative; max-width: 900px; margin: 8px 0;">
    <div id="fp2" style="background: #1b3139; border-radius: 8px; padding: 14px 18px; color: #e0e0e0; font-family: monospace; font-size: 13pt; line-height: 1.6; border-left: 4px solid #1B5162;">SELECT * FROM franchise_performance(NULL, 'Tokyo')</div>
    <div style="font-size: 13pt; color: #618794; margin: 4px 0 12px 0;">All franchises in Tokyo</div>
    <button onclick="var t=document.getElementById('fp2').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:8px;right:8px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <div style="position: relative; max-width: 900px; margin: 8px 0;">
    <div id="fp3" style="background: #1b3139; border-radius: 8px; padding: 14px 18px; color: #e0e0e0; font-family: monospace; font-size: 13pt; line-height: 1.6; border-left: 4px solid #1B5162;">SELECT * FROM franchise_performance(NULL, NULL)</div>
    <div style="font-size: 13pt; color: #618794; margin: 4px 0 0 0;">All 48 franchises worldwide, ranked by revenue</div>
    <button onclick="var t=document.getElementById('fp3').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:8px;right:8px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
</div>
</details>

#### B1c. Wrap as LangChain Tools

Now wrap both functions as LangChain-compatible tools so the agent can call them.

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Mini architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; border-left: 3px solid #FF6B35;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 2px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 2px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <div style="display: flex; justify-content: space-around; padding: 0 30px;"><div style="width: 2px; height: 8px; background: #1B3A4B;"></div><div style="width: 2px; height: 8px; background: #1B3A4B;"></div></div>
      <div style="border: 2px solid #1B3A4B; border-radius: 6px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; opacity: 0.3;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8;">System prompt &bull; ReAct loop</div>
      </div>
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto; opacity: 0.3;"></div>
      <div style="border: 1.5px solid #8B4513; border-radius: 4px; overflow: hidden; opacity: 0.3;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 4px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;"><div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div><div style="color: #666; font-size: 8pt;">Short-term</div></div>
          <div style="flex: 1; padding: 4px 6px; text-align: center; background: #FFF8F0;"><div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div><div style="color: #666; font-size: 8pt;">Long-term</div></div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden;">
      <div style="background: #1B5162; color: #fff; padding: 8px 14px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">UCFunctionToolkit</code></div>
      <div style="padding: 14px; background: #f8fafc; display: flex; gap: 10px; align-items: center;">
        <div style="flex: 1; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">UC Functions</div>
          <div style="font-size: 13pt; color: #333; margin-top: 4px;">SQL functions<br>in Unity Catalog</div>
        </div>
        <div style="font-size: 16pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1.2; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">UCFunctionToolkit</div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Wraps each function<br>as a LangChain tool</div>
        </div>
        <div style="font-size: 16pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1; text-align: center; padding: 10px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Agent-callable</div>
          <div style="font-size: 13pt; color: #333; margin-top: 4px;">LLM can invoke<br>these tools by name</div>
        </div>
      </div>
    </div>
  </div>
</div>


In [0]:
# UCFunctionToolkit wraps Unity Catalog SQL functions as LangChain-compatible tools
from databricks_langchain import UCFunctionToolkit

# Reference the two functions we just created using their fully-qualified UC names
uc_tool_names = [
    f"{catalog_name}.{schema_name}.top_products_by_franchise",
    f"{catalog_name}.{schema_name}.franchise_performance",
]

# Load and wrap them so the agent can call these functions as tools
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)
uc_tools = uc_toolkit.tools

print(f"Loaded UC tools: {[t.name for t in uc_tools]}")

### B2. Create Your Lakebase Instance

Both memory systems persist their data in **Lakebase**, Databricks' managed PostgreSQL service. In this step you will create a dedicated Lakebase Autoscaling project that will host both checkpoint tables (short-term) and the memory store (long-term).

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Compact architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <!-- Top row: Tools -->
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <!-- Arrows -->
      <div style="display: flex; justify-content: space-around; padding: 0 40px; opacity: 0.3;">
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto;"></div>
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto;"></div>
      </div>
      <!-- Agent -->
      <div style="border: 2px solid #1B3A4B; border-radius: 8px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; opacity: 0.3; margin-bottom: 2px;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8; margin-top: 2px;">System prompt &bull; ReAct loop</div>
      </div>
      <!-- Arrow to Lakebase -->
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto;"></div>
      <!-- Lakebase (highlighted) -->
      <div style="border: 2.5px solid #FF6B35; border-radius: 8px; overflow: hidden; border-left: 5px solid #FF6B35;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt; letter-spacing: 0.5px;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 8px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div>
            <div style="color: #666; font-size: 8pt;">Short-term</div>
          </div>
          <div style="flex: 1; padding: 8px 6px; text-align: center; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div>
            <div style="color: #666; font-size: 8pt;">Long-term</div>
          </div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; background: #fff;">
      <div style="background: #1B5162; color: #fff; padding: 10px 18px; font-size: 14pt; font-weight: 700;">Lakebase Autoscaling</div>
      <div style="padding: 16px 18px;">
        <ul style="margin: 0; padding: 0 0 0 18px; color: #333; font-size: 14pt; line-height: 1.8;">
          <li>Managed PostgreSQL by Databricks</li>
          <li>Native workspace authentication</li>
          <li>Both <code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">CheckpointSaver</code> and <code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">DatabricksStore</code> persist here</li>
          <li>Scale-to-zero when idle</li>
        </ul>
      </div>
    </div>
  </div>
</div>
Please see the <a href="https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_langchain.html#databricks_langchain.CheckpointSaver" style="color: #2574B5;">CheckpointSaver</a>
 and <a href="https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_langchain.html#databricks_langchain.DatabricksStore" style="color: #2574B5;">DatabricksStore</a> documentation for further reading.

#### B2a. Create Your Lakebase Autoscaling Project

Run the cell below to see your Lakebase project name, then follow the instructions to create it.

In [0]:
# Print the Lakebase project name to use
print(f"Your Lakebase project name: {lakebase_project_name}")
print(f"\nCopy and paste this name when creating the project below.")


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1; font-size: 14pt;">Create a Lakebase Autoscaling Project</strong>
<ol style="font-size: 14pt; margin: 12px 0 0 16px; color: #333; line-height: 1.6;">
<li style="font-size: 14pt; margin-bottom: 4px;">In the upper right, click the <strong style="font-size: 14pt;">Waffle icon</strong> and select <strong style="font-size: 14pt;">Lakebase Postgres</strong>.</li>
<li style="font-size: 14pt; margin-bottom: 4px;">In the <strong style="font-size: 14pt;">Autoscaling</strong> view, click <strong style="font-size: 14pt;">Create project</strong>.</li>
<li style="font-size: 14pt; margin-bottom: 4px;">In the <strong style="font-size: 14pt;">Create project</strong> dialog:
  <ul style="font-size: 14pt; margin: 8px 0 0 16px;">
    <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Project name:</strong> Paste the name printed above (e.g. <code style="font-size: 14pt;">bakehouse-sparky-mcspark</code>).</li>
    <li style="font-size: 14pt;"><strong style="font-size: 14pt;">Postgres version:</strong> Select <strong style="font-size: 14pt;">17</strong>.</li>
  </ul>
</li>
<li style="font-size: 14pt; margin-bottom: 4px;">Click <strong style="font-size: 14pt;">Create</strong> to provision the project.</li>
</ol>
<p style="font-size: 14pt; margin: 12px 0 0 0; color: #333;">Provisioning typically completes within seconds. The project will have a single <strong>production</strong> branch with autoscaling enabled.</p>
</div>


#### B2b. Verify Lakebase Project and Update App Configuration

Run the cell below to verify your Lakebase project was created successfully and update the `databricks.yml` with the database resource.

In [0]:
verify_lakebase_and_patch_bundle(lakebase_project_name, _app_dir)

### B3. Configure Memory Systems

With the Lakebase instance created, you will now initialize both memory systems and define the long-term memory tools. First, review how the two memory scopes differ, then set up each one.

#### Comparing Memory Scopes

Short-term and long-term memory differ along five key dimensions: **scope**, **storage mechanism**, **management style**, **typical use cases**, and **concrete examples**. Understanding these distinctions helps you design agents that use the right memory type for each situation.


<!-- ── Visual: two tall cards with accent borders ── -->
<style>
.d1-cards { display: flex; gap: 24px; margin: 20px auto; max-width: 960px; align-items: stretch; }
.d1-card { flex: 1; background: #fff; border-radius: 14px; overflow: hidden; box-shadow: 0 2px 16px rgba(27,49,57,0.10); border: 1.5px solid #E8E3DC; display: flex; flex-direction: column; }
.d1-card-top { padding: 20px 24px 14px; border-bottom: 1.5px solid #E8E3DC; }
.d1-card-top h3 { font-size: 16pt; font-weight: 700; color: #1B3139; margin: 0 0 2px 0; }
.d1-card-top .d1-mech { font-size: 14pt; font-weight: 600; }
.d1-card.teal .d1-card-top { border-left: 5px solid #2574B5; }
.d1-card.teal .d1-mech { color: #2574B5; }
.d1-card.green .d1-card-top { border-left: 5px solid #00A972; }
.d1-card.green .d1-mech { color: #00A972; }
.d1-card-body { padding: 16px 24px 24px; flex: 1; }
.d1-card-body ul { list-style: none; padding: 0; margin: 0; }
.d1-card-body li { padding: 10px 0; border-bottom: 1px solid #E8E3DC; font-size: 14pt; line-height: 1.5; color: #0b2026; }
.d1-card-body li:last-child { border-bottom: none; }
.d1-dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; margin-right: 10px; position: relative; top: -1px; }
.d1-card.teal .d1-dot { background: #2574B5; }
.d1-card.teal li strong { color: #2574B5; }
.d1-card.green .d1-dot { background: #00A972; }
.d1-card.green li strong { color: #00A972; }
.d1-card-body code { font-size: 14pt; padding: 2px 6px; border-radius: 4px; background: #F9F7F4; color: #1B5162; }
.d1-card-body .d1-example { font-style: italic; opacity: 0.85; }
</style>
<div class="d1-cards">
  <div class="d1-card teal">
    <div class="d1-card-top"><h3>Short-Term Memory</h3><div class="d1-mech">Checkpointers</div></div>
    <div class="d1-card-body"><ul>
      <li><span class="d1-dot"></span><strong>Scope:</strong> Single session (<code>thread_id</code>)</li>
      <li><span class="d1-dot"></span><strong>Storage:</strong> Full graph state: messages, tool calls, decisions</li>
      <li><span class="d1-dot"></span><strong>Management:</strong> Automatic. The checkpointer saves after each node</li>
      <li><span class="d1-dot"></span><strong>Use Case:</strong> Follow-ups, constraint recall, mid-task recovery</li>
      <li><span class="d1-dot"></span><strong>Example:</strong> <span class="d1-example">"Only private rooms under $200" &#x2192; filters current results</span></li>
    </ul></div>
  </div>
  <div class="d1-card green">
    <div class="d1-card-top"><h3>Long-Term Memory</h3><div class="d1-mech">Store</div></div>
    <div class="d1-card-body"><ul>
      <li><span class="d1-dot"></span><strong>Scope:</strong> Across all sessions (<code>user_id</code>)</li>
      <li><span class="d1-dot"></span><strong>Storage:</strong> Curated facts: preferences, learned behaviors</li>
      <li><span class="d1-dot"></span><strong>Management:</strong> Explicit. The agent calls <code>save_memory</code></li>
      <li><span class="d1-dot"></span><strong>Use Case:</strong> Personalization, preference recall, progressive learning</li>
      <li><span class="d1-dot"></span><strong>Example:</strong> <span class="d1-example">"Prefers Golden Gate Ginger reports" &#x2192; recalled next session</span></li>
    </ul></div>
  </div>
</div>

<br/>

<style>
details summary span:first-child {
transition: transform 0.2s ease;
display: inline-block;
}
details[open] summary span:first-child {
transform: rotate(90deg);
}
</style>



#### B3a. Initialize Short-Term Memory (CheckpointSaver)

Short-term memory saves automatically after every node in the agent graph. You just need to create the checkpoint tables in Lakebase.

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Compact architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <!-- Top row: Tools -->
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 2px solid #1B5162; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; border-left: 5px solid #FF6B35;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <!-- Arrows -->
      <div style="display: flex; justify-content: space-around; padding: 0 40px;">
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto; opacity: 0.3;"></div>
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto;"></div>
      </div>
      <!-- Agent -->
      <div style="border: 2px solid #1B3A4B; border-radius: 8px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; opacity: 0.3; margin-bottom: 2px;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8; margin-top: 2px;">System prompt &bull; ReAct loop</div>
      </div>
      <!-- Arrow to Lakebase -->
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto; opacity: 0.3;"></div>
      <!-- Lakebase -->
      <div style="border: 1.5px solid #8B4513; border-radius: 8px; overflow: hidden; opacity: 0.3;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt; letter-spacing: 0.5px;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 8px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div>
            <div style="color: #666; font-size: 8pt;">Short-term</div>
          </div>
          <div style="flex: 1; padding: 8px 6px; text-align: center; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div>
            <div style="color: #666; font-size: 8pt;">Long-term</div>
          </div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; background: #fff;">
      <div style="background: #1B5162; color: #fff; padding: 10px 18px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">CheckpointSaver</code> <span style="font-weight: 400; color: #a8c9d6; font-size: 13pt;">- short-term</span></div>
      <div style="padding: 14px 18px; display: flex; align-items: center; gap: 10px;">
        <div style="flex: 1; text-align: center; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Automatic</div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Saves full graph state<br>after every node</div>
        </div>
        <div style="font-size: 18pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1; text-align: center; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">Scoped by <code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">thread_id</code></div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Each conversation<br>isolated per thread</div>
        </div>
      </div>
    </div>
  </div>
</div>

In [0]:
import uuid
from databricks.sdk import WorkspaceClient
from databricks_langchain import DatabricksStore, CheckpointSaver

# ----- Initialize checkpoint tables -----
# Creates the 'checkpoints', 'checkpoint_blobs', 'checkpoint_migrations', 'checkpoint_writes' tables
with CheckpointSaver(
    project=lakebase_autoscaling_project,
    branch=lakebase_autoscaling_branch,
) as saver:
    saver.setup()
print("CheckpointSaver (short-term memory) initialized.")


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
<strong style="color:#2e7d32;">&#10003; Verify in Lakebase</strong>
<ol style="margin: 8px 0 0 0; color: #333; line-height: 1.7;">
  <li>Click the <strong>Waffle icon</strong> in the upper right and select <strong>Lakebase Postgres</strong> > <strong>Autoscaling</strong>.</li>
  <li>Click on your <strong>bakehouse</strong> project, then click <strong>Tables</strong> in the left menu.</li>
  <li>You should see <strong>4 new tables</strong> in the <code>public</code> schema:</li>
</ol>
<ul style="margin: 4px 0 0 24px; color: #333;">
  <li><code>checkpoints</code> - full conversation state snapshots, one per graph step</li>
  <li><code>checkpoint_blobs</code> - serialized message payloads for each checkpoint</li>
  <li><code>checkpoint_writes</code> - pending writes during node transitions</li>
  <li><code>checkpoint_migrations</code> - schema version tracking</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;">All 4 tables are <strong>empty</strong> right now. They will be populated once you start running conversations in Section D. These tables are what allow the agent to remember earlier turns in the same conversation thread.</p>
</div>

#### B3b. Initialize Long-Term Memory (DatabricksStore)

Long-term memory requires both a store (for persistence) and tools (for the agent to read and write). First initialize the store tables, then define the tools.

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Compact architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <!-- Top row: Tools -->
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 2px solid #1B5162; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; border-left: 5px solid #FF6B35;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <!-- Arrows -->
      <div style="display: flex; justify-content: space-around; padding: 0 40px;">
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto; opacity: 0.3;"></div>
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto;"></div>
      </div>
      <!-- Agent -->
      <div style="border: 2px solid #1B3A4B; border-radius: 8px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; opacity: 0.3; margin-bottom: 2px;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8; margin-top: 2px;">System prompt &bull; ReAct loop</div>
      </div>
      <!-- Arrow to Lakebase -->
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto; opacity: 0.3;"></div>
      <!-- Lakebase -->
      <div style="border: 1.5px solid #8B4513; border-radius: 8px; overflow: hidden; opacity: 0.3;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt; letter-spacing: 0.5px;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 8px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div>
            <div style="color: #666; font-size: 8pt;">Short-term</div>
          </div>
          <div style="flex: 1; padding: 8px 6px; text-align: center; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div>
            <div style="color: #666; font-size: 8pt;">Long-term</div>
          </div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <!-- save_memory box -->
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; background: #fff; margin-bottom: 12px;">
      <div style="background: #1B5162; color: #fff; padding: 10px 18px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">save_memory(key, value)</code> <span style="font-weight: 400; color: #a8c9d6; font-size: 13pt;">- long-term</span></div>
      <div style="padding: 14px 18px; display: flex; align-items: center; gap: 10px;">
        <div style="flex: 1; text-align: center; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 600; color: #1B3139;">Agent learns a fact</div>
          <div style="font-size: 13pt; color: #618794; font-style: italic; margin-top: 4px;">"Manager prefers weekly reports"</div>
        </div>
        <div style="font-size: 18pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1; text-align: center; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">store.put()</div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Persists to Lakebase<br>scoped by <code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">user_id</code></div>
        </div>
      </div>
    </div>
    <!-- recall_memories box -->
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; background: #fff;">
      <div style="background: #1B5162; color: #fff; padding: 10px 18px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">recall_memories(query)</code> <span style="font-weight: 400; color: #a8c9d6; font-size: 13pt;">- long-term</span></div>
      <div style="padding: 14px 18px; display: flex; align-items: center; gap: 10px;">
        <div style="flex: 1; text-align: center; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">store.search()</div>
          <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Searches by relevance within<br><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">("managers", user_id)</code></div>
        </div>
        <div style="font-size: 18pt; color: #1B5162; font-weight: 700;">&#x2192;</div>
        <div style="flex: 1; text-align: center; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
          <div style="font-size: 14pt; font-weight: 600; color: #1B3139;">Top matches by meaning</div>
          <div style="font-size: 13pt; color: #618794; font-style: italic; margin-top: 4px;">[name]: Jordan Chen<br>[franchise]: Golden Crumbs</div>
        </div>
      </div>
    </div>
  </div>
</div>

In [0]:
# ----- Long-term memory store -----
# Creates the 'store' and 'store_migrations' tables in Lakebase
store = DatabricksStore(
    project=lakebase_autoscaling_project,
    branch=lakebase_autoscaling_branch,
    workspace_client=WorkspaceClient(),
    embedding_endpoint=EMBEDDING_ENDPOINT,
    embedding_dims=EMBEDDING_DIMS,
)
store.setup()
print("DatabricksStore (long-term memory) initialized.")


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
<strong style="color:#2e7d32;">&#10003; Verify in Lakebase</strong>
<p style="margin: 8px 0 0 0; color: #333;">Go back to the <strong>Tables</strong> page in your Lakebase project and refresh. You should now see <strong>4 additional tables</strong> (8 total):</p>
<ul style="margin: 4px 0 0 24px; color: #333;">
  <li><code>store</code> - key-value facts organized by namespace (e.g., per-manager preferences)</li>
  <li><code>store_migrations</code> - schema version tracking for the store</li>
  <li><code>store_vectors</code> - embedding vectors for similarity search</li>  
  <li><code>vector_migrations</code> - schema version tracking for the vector table</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;">Like the checkpoint tables, these are <strong>empty</strong> for now. The agent will populate them when it calls <code>save_memory</code> during conversations. Both memory systems share this single Lakebase instance but use separate tables.</p>
</div>

In [0]:
from langchain_core.tools import tool               # @tool decorator makes a function callable by the agent
from langchain_core.runnables import RunnableConfig  # Carries thread_id and user_id through the graph
from langgraph.store.base import BaseStore           # Base class for the long-term memory store
from typing import Annotated
from langgraph.prebuilt import InjectedStore         # Tells LangGraph to auto-inject the store at runtime


@tool
def save_memory(
    key: str,                                        # The LLM provides key and value
    value: str,
    config: RunnableConfig,                          # LangGraph injects this; contains user_id
    store: Annotated[BaseStore, InjectedStore],      # LangGraph injects this; the DatabricksStore instance
) -> str:
    """Save an important fact about this franchise manager to long-term memory.

    Args:
        key:   A short label like 'name', 'franchise', 'preferred_product', 'reporting_preference'.
        value: The fact to remember, e.g. 'manages Golden Crumbs in San Francisco'.
    """
    # Scope memories to this specific manager using their user_id
    user_id = config["configurable"]["user_id"]
    namespace = ("managers", user_id)

    # Write the fact to the store; overwrites if the same key already exists
    store.put(namespace, key, {"content": value})
    return f"Saved '{key}' = '{value}' to long-term memory."


@tool
def recall_memories(
    query: str,                                      # The LLM provides the search query
    config: RunnableConfig,                          # LangGraph injects this; contains user_id
    store: Annotated[BaseStore, InjectedStore],      # LangGraph injects this; the DatabricksStore instance
) -> str:
    """Search this manager's long-term memory for relevant facts.

    Args:
        query: What you're looking for, e.g. 'manager preferences', 'franchise details'.
    """
    # Search only this manager's namespace; other managers' data is invisible
    user_id = config["configurable"]["user_id"]
    namespace = ("managers", user_id)

    # Semantic search: finds facts by meaning, not exact key match
    results = store.search(namespace, query=query, limit=5)

    if not results:
        return "No memories found for this manager."

    lines = [f"  [{item.key}]: {item.value['content']}" for item in results]
    return "Recalled memories:\n" + "\n".join(lines)


print("Memory tools defined: save_memory, recall_memories")


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Key Design Decisions</strong>
  <ul style="margin-bottom: 0;">
    <li>The <strong>namespace</strong> is <code>("managers", user_id)</code>: each franchise manager gets their own isolated memory space.</li>
    <li><strong>save_memory</strong> uses <code>store.put()</code> for exact key-value storage. If the same key is saved again, it overwrites the previous value.</li>
    <li><strong>recall_memories</strong> uses <code>store.search()</code> for semantic similarity search, so the agent does not need to remember exact keys.</li>
  </ul>
</div>


### B4. Define the Agent Graph

Now build the LangGraph agent that ties all four tools together. The system prompt instructs the agent to proactively use memory at the start of every conversation.

<div style="display: flex; gap: 24px; align-items: center; max-width: 100%; margin: 16px auto; font-family: -apple-system, sans-serif;">
  <!-- LEFT: Compact architecture diagram -->
  <div style="flex: 0 0 260px;">
    <div style="font-size: 9pt;">
      <!-- Top row: Tools -->
      <div style="display: flex; gap: 4px; margin-bottom: 2px;">
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">UC Function Tools</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">top_products_by_franchise</code><br><code style="font-size: 8pt;">franchise_performance</code></div>
        </div>
        <div style="flex: 1; border: 1.5px solid #1B3A4B; border-radius: 4px; padding: 4px 6px; background: #f8fafc; text-align: center; opacity: 0.3;">
          <div style="font-weight: 700; color: #1B3A4B; font-size: 9pt;">Agent Memory</div>
          <div style="color: #666; font-size: 8pt; margin-top: 3px;"><code style="font-size: 8pt;">CheckpointSaver</code><br><code style="font-size: 8pt;">save / recall</code></div>
        </div>
      </div>
      <!-- Arrows -->
      <div style="display: flex; justify-content: space-around; padding: 0 40px;">
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto;"></div>
        <div style="width: 2px; height: 8px; background: #1B3A4B; margin: 0 auto;"></div>
      </div>
      <!-- Agent (highlighted) -->
      <div style="border: 2.5px solid #FF6B35; border-radius: 8px; padding: 6px 8px; background: #1B3A4B; color: white; text-align: center; border-left: 5px solid #FF6B35; margin-bottom: 2px;">
        <div style="font-weight: 700; font-size: 10pt;">Agent</div>
        <div style="font-size: 8pt; opacity: 0.8; margin-top: 2px;">System prompt &bull; ReAct loop</div>
      </div>
      <!-- Arrow to Lakebase -->
      <div style="width: 2px; height: 8px; background: #8B4513; margin: 0 auto; opacity: 0.3;"></div>
      <!-- Lakebase -->
      <div style="border: 1.5px solid #8B4513; border-radius: 8px; overflow: hidden; opacity: 0.3;">
        <div style="background: #8B4513; color: white; text-align: center; padding: 4px; font-weight: 600; font-size: 9pt; letter-spacing: 0.5px;">LAKEBASE</div>
        <div style="display: flex;">
          <div style="flex: 1; padding: 8px 6px; text-align: center; border-right: 1px solid #E8D5C4; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">CheckpointSaver</div>
            <div style="color: #666; font-size: 8pt;">Short-term</div>
          </div>
          <div style="flex: 1; padding: 8px 6px; text-align: center; background: #FFF8F0;">
            <div style="font-weight: 700; color: #8B4513; font-size: 8pt;">DatabricksStore</div>
            <div style="color: #666; font-size: 8pt;">Long-term</div>
          </div>
        </div>
      </div>
    </div>
  </div>
  <!-- Arrow connector -->
  <div style="font-size: 24pt; color: #1B5162;">&#x2192;</div>
  <!-- RIGHT: Detail panel -->
  <div style="flex: 1;">
    <div style="border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; background: #fff;">
      <div style="background: #1B5162; color: #fff; padding: 8px 14px; font-size: 14pt; font-weight: 700;">Building the Graph</div>
      <div style="padding: 12px 14px;">
        <div style="display: flex; gap: 8px; align-items: stretch;">
          <div style="flex: 1; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
            <div style="font-size: 13pt; font-weight: 700; color: #1B5162;">SYSTEM_PROMPT</div>
            <div style="font-size: 12pt; color: #618794; margin-top: 4px;">Instructs agent to<br>recall at start, save<br>when learning facts</div>
          </div>
          <div style="flex: 1; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
            <div style="font-size: 13pt; font-weight: 700; color: #1B5162;">all_tools</div>
            <div style="font-size: 12pt; color: #618794; margin-top: 4px;"><code style="font-size: 11pt; background: #EEEDE9; padding: 1px 3px; border-radius: 2px;">uc_tools</code> +<br><code style="font-size: 11pt; background: #EEEDE9; padding: 1px 3px; border-radius: 2px;">save_memory</code><br><code style="font-size: 11pt; background: #EEEDE9; padding: 1px 3px; border-radius: 2px;">recall_memories</code></div>
          </div>
          <div style="flex: 1; padding: 10px; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 6px;">
            <div style="font-size: 13pt; font-weight: 700; color: #1B5162;">llm_with_tools</div>
            <div style="font-size: 12pt; color: #618794; margin-top: 4px;">ChatDatabricks<br>with tools bound</div>
          </div>
        </div>
        <div style="text-align: center; color: #1B5162; font-size: 14pt; margin: 6px 0;">&#x25BC;</div>
        <div style="display: flex; gap: 6px; align-items: center; justify-content: center; padding: 8px 12px; background: #1B3139; border-radius: 6px; color: #ccc; font-family: Consolas, monospace; font-size: 12pt;">
          <span style="color: #f08d49;">START</span> &#x2192;
          <span style="color: #7ec699;">agent_node</span> &#x2192;
          <span style="color: #cc99cd;">should_continue?</span> &#x2192;
          <span style="color: #7ec699;">tool_node</span> &#x2192;
          <span style="color: #7ec699;">agent_node</span> &#x2192;
          <span style="color: #f08d49;">END</span>
        </div>
      </div>
    </div>
  </div>
</div>

In [0]:
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# ----- Combine all tools -----
all_tools = uc_tools + [save_memory, recall_memories]

# ----- Initialize the LLM -----
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
llm_with_tools = llm.bind_tools(all_tools)

# ----- Define agent and tool nodes -----
def agent_node(state: MessagesState):
    """The agent node: adds system prompt and calls the LLM."""
    system_message = {"role": "system", "content": SYSTEM_PROMPT}
    response = llm_with_tools.invoke([system_message] + state["messages"])
    return {"messages": [response]}


tool_node = ToolNode(all_tools)

# ----- Routing function -----
def should_continue(state: MessagesState):
    """Route to tools if the LLM requested tool calls, otherwise end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# ----- Build the graph -----
workflow = StateGraph(MessagesState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, ["tools", END])
workflow.add_edge("tools", "agent")

print("Agent graph defined with nodes: agent, tools")
print(f"Total tools: {len(all_tools)} ({[t.name for t in all_tools]})")


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Key Design Decisions</strong>
  <ul style="margin-bottom: 0;">
    <li>The <strong>system prompt</strong> instructs the agent to call <code>recall_memories</code> at the start of every conversation and <code>save_memory</code> when it learns new facts. The agent decides when and what to remember.</li>
    <li><strong>all_tools</strong> combines both UC function tools (data queries) and memory tools (save/recall) into a single list. The agent chooses which to call based on the user's request.</li>
    <li>The <strong>ReAct loop</strong> (<code>agent > tools > agent</code>) lets the agent call multiple tools per turn. It may recall memories, query data, and save new facts in a single response cycle.</li>
    <li>The graph is defined but <strong>not yet compiled</strong>. Compilation with <code>checkpointer</code> and <code>store</code> happens in the helper function, which opens a fresh Lakebase connection per invocation.</li>
  </ul>
</div>

## C. Connect and Prepare to Test

With the full agent assembled, the last step before testing is to create the helper functions that connect the graph to Lakebase.


### C1. Helper Functions

The `run_agent` function is the single entry point for every conversation turn. It connects the graph you defined in B4 to the Lakebase memory you initialized in B3.

<div style="max-width: 960px; margin: 16px auto; border: 2px solid #1B5162; border-radius: 10px; overflow: hidden; font-family: -apple-system, sans-serif;">
  <div style="background: #1B5162; color: #fff; padding: 10px 18px; font-size: 14pt; font-weight: 700;"><code style="font-size: 14pt; color: #a8c9d6; background: none;">run_agent(query, thread_id, user_id)</code></div>
  <div style="padding: 14px 18px; background: #f8fafc; display: flex; gap: 10px; align-items: stretch;">
    <div style="flex: 1; padding: 12px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px;">
      <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">1. Pack config</div>
      <div style="font-size: 13pt; color: #618794; margin-top: 4px;"><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">thread_id</code> scopes short-term<br><code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">user_id</code> scopes long-term</div>
    </div>
    <div style="font-size: 16pt; color: #1B5162; font-weight: 700; display: flex; align-items: center;">&#x2192;</div>
    <div style="flex: 1; padding: 12px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px;">
      <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">2. Compile graph</div>
      <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Opens CheckpointSaver<br>and compiles with <code style="font-size: 13pt; background: #EEEDE9; padding: 1px 4px; border-radius: 3px;">store</code></div>
    </div>
    <div style="font-size: 16pt; color: #1B5162; font-weight: 700; display: flex; align-items: center;">&#x2192;</div>
    <div style="flex: 1; padding: 12px; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 8px;">
      <div style="font-size: 14pt; font-weight: 700; color: #1B5162;">3. Invoke</div>
      <div style="font-size: 13pt; color: #618794; margin-top: 4px;">Runs the ReAct loop<br>returns final response</div>
    </div>
  </div>
</div>

The `show` function renders the agent's markdown-formatted responses.

In [0]:
from IPython.display import Markdown, display

def show(text):
    """Render agent response as formatted markdown."""
    display(Markdown(text))


# ----- Helper function to run the agent -----
def run_agent(query, thread_id, user_id="manager_jordan_sf"):
    """Run the Bakehouse agent with the given query, thread, and user identity."""

    # Step 1: Pack config with both scoping keys
    config = {
        "configurable": {
            "thread_id": thread_id,   # short-term memory scope
            "user_id": user_id,       # long-term memory scope
        }
    }

    # Step 2: Open CheckpointSaver connection and compile graph with both memory systems
    with CheckpointSaver(
        project=lakebase_autoscaling_project,
        branch=lakebase_autoscaling_branch,
    ) as checkpointer:
        graph = workflow.compile(checkpointer=checkpointer, store=store)

        # Step 3: Invoke the ReAct loop and return the final response
        result = graph.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config,
        )
    return result["messages"][-1].content


print("run_agent() helper ready. You can now test the agent!")

## D. Test the Agent

### D1. Test Short-Term Memory

In [0]:
session_thread = str(uuid.uuid4())
print(f"Session thread ID: {session_thread}")

#### D1a. Turn 1: Introduce yourself, share preferences, and ask about top products

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Turn 1</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">Introduce yourself</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #2574B5; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">session_thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #00A972; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">manager_jordan_sf</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">Empty - first turn in thread</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">Empty - no facts stored yet</div>
    </div>
  </div>
</div>

In [0]:
query_1 = """Hi there! I'm Jordan Chen and I manage the Golden Crumbs franchise in San Francisco.
I prefer weekly sales summaries and I'm especially interested in tracking our Austin Almond Biscotti sales
because it's our signature product. Can you pull up our top products?"""

response_1 = run_agent(query_1, thread_id=session_thread)
show(response_1)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul style="margin-bottom: 0;">
    <li>The agent should call <code>recall_memories</code> first. On a first run, it finds nothing. On a re-run, it may find memories from a prior execution.</li>
    <li>It should call <code>save_memory</code> multiple times to store Jordan's name, franchise, preferred product (Austin Almond Biscotti), and reporting preference (weekly summaries).</li>
    <li>It should call <code>top_products_by_franchise</code> with "Golden Crumbs" and show Austin Almond Biscotti as the top product.</li>
    <li>Check your <strong>MLflow traces</strong> to see the full sequence: recall, save(s), data query, response.</li>
  </ul>
</div>


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Pause and Explore: What Just Happened in Lakebase?</strong>
  <p>That single query triggered <strong>three agent actions</strong>: recall existing memories, query sales data, and save new memories. All of this is now persisted in Lakebase. Take a few minutes to explore before continuing.</p>

  <strong>1. Review the MLflow Trace</strong>
  <ul>
    <li>In the cell output above, expand the <strong>MLflow Trace</strong>.</li>
    <li>You should see the agent's ReAct loop, with multiple rounds of <code>agent &rarr; tools &rarr; agent</code>. Look for calls to <code>recall_memories</code>, a data query tool, and <code>save_memory</code>.</li>
    <li>On a first run, <code>recall_memories</code> returns nothing. The agent then saves Jordan's details and queries the data. On a re-run, recall may find existing memories.</li>
  </ul>

  <strong>2. Examine the <code>checkpoint_writes</code> Table (Short-Term Memory)</strong>
  <ul>
    <li>Open your <strong>Lakebase project</strong> (you may still have it open in another tab from earlier) and navigate to <strong>Tables &rarr; checkpoint_writes</strong>.</li>
    <li>Each row is a write operation from one step of the agent graph. Look at the <code>channel</code> and <code>blob</code> columns:
      <ul>
        <li><code>channel</code>: you will see <code>messages</code> rows (conversation content) and routing rows like <code>branch:to:agent</code> or <code>branch:to:tools</code></li>
        <li><code>blob</code>: the serialized payload for each step, stored as bytea. The content is not human-readable, but the pattern of rows shows you the flow.</li>
      </ul>
    </li>
    <li>All rows share the same <code>thread_id</code>: this is how short-term memory is scoped to a single conversation. The number of rows will grow as you continue the conversation.</li>
  </ul>

  <strong>3. Examine the <code>store</code> Table (Long-Term Memory)</strong>
  <ul>
    <li>Switch to the <code>store</code> table in Lakebase.</li>
    <li>You should see multiple rows the agent saved (name, franchise, preferred product, reporting preference). The exact keys vary by run, but the key columns are:
      <ul>
        <li><code>prefix</code>: the namespace scoped to this manager</li>
        <li><code>key</code>: a descriptive key the agent chose when calling <code>save_memory</code></li>
        <li><code>value</code>: a JSON object with the stored fact</li>
      </ul>
    </li>
    <li>This is the <strong>long-term memory</strong> that will persist across sessions. Even if we start a completely new thread, the agent can recall these facts.</li>
  </ul>

  <p>One query wrote to both tables. The <code>checkpoint_writes</code> table captures the full conversation flow (short-term), while the <code>store</code> table captures curated facts about the user (long-term). As you continue the conversation, watch both tables grow.</p>
</div>

#### D1b. Turn 2: Ask a follow-up that requires context from Turn 1

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Turn 2</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">Follow-up comparison</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #2574B5; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">session_thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #00A972; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">manager_jordan_sf</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">1 turn: name, franchise, top products</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">Agent may have saved: name, franchise</div>
    </div>
  </div>
</div>

In [0]:
query_2 = "How does that compare to other franchises in the US?"

response_2 = run_agent(query_2, thread_id=session_thread)
show(response_2)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe: Short-Term Memory in Action</strong>
  <p>The agent understands "that" refers to <strong>Golden Crumbs' product performance</strong> from Turn 1. It calls <code>franchise_performance</code> or <code>top_products_by_franchise</code> filtered to the US to compare. This context comes from <strong>short-term memory</strong>: the checkpoint preserves the full conversation history within this thread.</p>
</div>

#### D1c. Turn 3: Drill deeper into franchise comparison

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Turn 3</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">Deeper analysis</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #2574B5; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">session_thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #00A972; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">manager_jordan_sf</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">2 turns: + US franchise comparison</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">Saving facts as conversation continues</div>
    </div>
  </div>
</div>

In [0]:
query_3 = "Which of those franchises has the highest average order value?"

response_3 = run_agent(query_3, thread_id=session_thread)
show(response_3)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <p>The agent knows "those franchises" means the US franchises from Turn 2. It can answer using data already retrieved or by making a new targeted query. Either way, <strong>short-term memory</strong> provides the conversational context.</p>
</div>

#### D1d. Turn 4: Request a recommendation based on accumulated context

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Turn 4</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">Personalized advice</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #2574B5; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">session_thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #00A972; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">manager_jordan_sf</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">3 turns: + avg order analysis</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">Multiple facts stored for this manager</div>
    </div>
  </div>
</div>

In [0]:
query_4 = "Based on what you know about my franchise, what product should I focus on promoting?"

response_4 = run_agent(query_4, thread_id=session_thread)
show(response_4)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <p>The agent should provide a personalized recommendation for <strong>Golden Crumbs in San Francisco</strong>, using data from all previous turns. This demonstrates how short-term memory enables <strong>multi-turn reasoning</strong> within a single session.</p>
</div>

### D2. Demonstrating Stateless Behavior

Compare stateful vs stateless behavior by running the **exact same query** from Turn 4, but this time with a **completely fresh, stateless session**.

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-left: 5px solid #98102A; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Stateless</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">Same query, no memory</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #FF6B35; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">stateless_thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #98102A; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">anonymous</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">Empty - brand new thread</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">Empty - unknown user ID</div>
    </div>
  </div>
</div>

In [0]:
# Stateless session: no memory of any kind
stateless_thread = str(uuid.uuid4())
response_stateless = run_agent(query_4, thread_id=stateless_thread, user_id="anonymous_no_memory")
show(response_stateless)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <p>Without any memory, the agent cannot answer the question. It should ask for clarification or provide a generic answer.</p>
  <p>Compare this to Turn 4 above: same query, completely different result. In the MLflow trace, compare the token count: Turn 4 included the full 4-turn conversation history, while this trace includes only the system prompt and the single query.</p>
  <p>This also demonstrates that memory is <strong>scoped per user</strong>. A different <code>user_id</code> means a different namespace in the store. Even if Jordan's facts are saved in Lakebase, an anonymous user cannot access them. Each identity gets its own isolated memory space.</p>
</div>

### D3. Test Cross-Session Memory

Cross-session memory (powered by **DatabricksStore**) lets the agent remember facts about a manager even in a **brand-new conversation thread**. This is a key difference between short-term and long-term memory.

#### D3a. Test Recall: Does the Agent Remember?

Start a completely new thread with the same `user_id`. The manager does NOT re-introduce themselves. Instead, they ask a casual question that only makes sense if the agent recalls their stored facts (name, franchise, signature product).

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">New Session</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">Test long-term recall</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #FF6B35; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">new thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #00A972; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">manager_jordan_sf</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">Empty - new thread starts fresh</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">recall should find: name, franchise, preferences from D1</div>
    </div>
  </div>
</div>

In [0]:
new_thread = str(uuid.uuid4())
manager_user_id = "manager_jordan_sf"

print(f"Session 2 Thread: {new_thread}")
print(f"Manager User ID:  {manager_user_id}")

In [0]:
intro_query = "Hey, I'd like to check on how my signature product is doing this quarter. Can you pull the numbers?"

response_s1 = run_agent(intro_query, thread_id=new_thread, user_id=manager_user_id)
show(response_s1)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul style="margin-bottom: 0;">
    <li>The agent should call <code>recall_memories</code> first and find the facts saved during D1 (name, franchise, preferred product, reporting preference).</li>
    <li>It should <strong>greet Jordan by name</strong> and reference <strong>Golden Crumbs</strong> without being told. The query says "my signature product" and the agent should know that means Austin Almond Biscotti from stored facts.</li>
    <li>Finally, it calls <code>top_products_by_franchise</code> to answer the data question.</li>
  </ul>
</div>

#### D3b. Update a Preference (New Thread)

Another new thread, same manager. The agent should recall Jordan's profile, then update the preferred product when asked. This tests both recall AND the ability to overwrite stored facts via `save_memory`.

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-left: 5px solid #00A972; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Update Preference</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">New thread, update stored fact</div>
  </div>
  <div style="flex: 0 0 auto; display: flex; flex-direction: column; gap: 4px;">
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #FF6B35; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">new thread</div>
    <div style="font-family: Consolas, monospace; font-size: 13pt; background: #00A972; color: #fff; padding: 3px 10px; border-radius: 4px; white-space: nowrap;">manager_jordan_sf</div>
  </div>
  <div style="flex: 1; display: flex; gap: 8px;">
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #2574B5; margin-bottom: 2px;">Short-term</div>
      <div style="font-size: 13pt; color: #333;">Empty - new thread again</div>
    </div>
    <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 10px;">
      <div style="font-size: 12pt; font-weight: 600; color: #00A972; margin-bottom: 2px;">Long-term</div>
      <div style="font-size: 13pt; color: #333;">Recall first, then overwrite: Biscotti to Golden Gate Ginger</div>
    </div>
  </div>
</div>


In [0]:
update_thread = str(uuid.uuid4())
update_query = "Actually, I've been really impressed with our Golden Gate Ginger sales lately. Let's make that my preferred product instead of the Biscotti."

response_update = run_agent(update_query, thread_id=update_thread, user_id=manager_user_id)
show(response_update)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <p><strong>In the output:</strong> The agent should acknowledge the preference change from Austin Almond Biscotti to Golden Gate Ginger.</p>
  <p><strong>In the MLflow trace:</strong> Look for a <code>recall_memories</code> call that retrieves Jordan's profile, followed by a <code>save_memory</code> call with "Golden Gate Ginger". Because <code>store.put()</code> overwrites existing keys, the old Biscotti preference should be replaced.</p>
  <p><strong>Note:</strong> The exact key name depends on the LLM's choice. If the agent uses a different key (e.g., "favorite_product" instead of "preferred_product"), the old entry will coexist rather than be overwritten. The Inspect Store step below shows what actually happened.</p>
</div>

#### D3c. Inspect the Store

Let's directly examine what the agent saved to long-term memory.

<div style="max-width: 960px; margin: 14px auto; font-family: -apple-system, sans-serif; background: #F9F7F4; border: 1.5px solid #DCE0E2; border-radius: 8px; padding: 12px 16px; display: flex; gap: 16px; align-items: center;">
  <div style="flex: 0 0 auto; min-width: 160px;">
    <div style="font-size: 14pt; font-weight: 700; color: #1B3139;">Direct lookup</div>
    <div style="font-size: 13pt; color: #618794; margin-top: 2px;">What did the agent save?</div>
  </div>
  <div style="flex: 1; background: #fff; border: 1.5px solid #DCE0E2; border-radius: 6px; padding: 8px 14px;">
    <div style="font-size: 14pt; color: #333;"><code style="font-size: 13pt; background: #EEEDE9; padding: 2px 6px; border-radius: 4px;">store.search()</code> returns all facts saved under <code style="font-size: 13pt; background: #EEEDE9; padding: 2px 6px; border-radius: 4px;">manager_jordan_sf</code></div>
  </div>
</div>


In [0]:
saved_memories = store.search(("managers", manager_user_id), query="all manager information", limit=10)

print(f"Long-term memories for '{manager_user_id}':\n")
for item in saved_memories:
    print(f"  [{item.key}]: {item.value['content']}")


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <p>You should see key-value pairs stored under the <code>managers/manager_jordan_sf</code> namespace. The agent decides how to organize the facts; it may store name and franchise as separate keys or combine them into a single profile entry. Look for entries covering:</p>
  <ul>
    <li>Jordan's identity and franchise (name, Golden Crumbs, San Francisco)</li>
    <li>Preferred product: Golden Gate Ginger (updated in the previous step)</li>
    <li>Reporting preference: weekly sales summaries</li>
  </ul>
  <p>The exact keys and number of entries vary by run. The LLM decides how to label and group each fact.</p>
</div>

## E. Prepare for Production

You have built and tested a stateful agent in a notebook. The next step is to serve it as a production application that franchise managers can access from any browser. Databricks Apps provides a managed runtime for Python web applications with built-in access to workspace resources like serving endpoints, MLflow experiments, and Lakebase.


### E1. Databricks Apps: Architecture and Resources

The `app.yaml` file declares which workspace resources the app can access. Each resource binding grants the app's service principal exactly the permissions it needs.


<!-- Combined D1 Architecture + D2 Resource Bindings visual -->
<style>
.d6t input[type="radio"] { display: none; }
.d6t-tabs { display: flex; gap: 0; margin-bottom: 0; }
.d6t-tab { flex: 1; text-align: center; padding: 14px 10px; font-size: 14pt !important; font-weight: 700; color: #5A6F77; cursor: pointer; border: 2px solid #DCE0E2; background: #fff; border-bottom: none; user-select: none; transition: all 0.2s; }
.d6t-tab:first-child { border-radius: 10px 0 0 0; }
.d6t-tab:last-child { border-radius: 0 10px 0 0; border-left: none; }
#d6t1:checked ~ .d6t-tabs .d6t-tab[for="d6t1"],
#d6t2:checked ~ .d6t-tabs .d6t-tab[for="d6t2"] { background: #F9F7F4; border-bottom-color: #F9F7F4; border-top: 3px solid #1B5162; color: #1B5162; }
.d6t-panels { border: 2px solid #DCE0E2; border-top: none; border-radius: 0 0 12px 12px; background: #F9F7F4; }
.d6t-panel { display: none; }
#d6t1:checked ~ .d6t-panels .d6t-p1,
#d6t2:checked ~ .d6t-panels .d6t-p2 { display: block; }
/* D1 architecture stack styles */
.d6t .d1v-wrap { border: 2px solid #DCE0E2; border-radius: 12px; overflow: hidden; background: #fff; margin: 16px; }
.d6t .d1v-legend { display: flex; gap: 24px; justify-content: center; padding: 14px 24px; background: #F9F7F4; border-bottom: 2px solid #E8E3DC; }
.d6t .d1v-leg { display: flex; align-items: center; gap: 8px; font-size: 15pt; font-weight: 700; }
.d6t .d1v-dot { width: 14px; height: 14px; border-radius: 3px; }
.d6t .d1v-stack { padding: 20px 24px; display: flex; flex-direction: column; gap: 0; }
.d6t .d1v-layer { display: flex; align-items: stretch; border: 2px solid #DCE0E2; border-radius: 8px; overflow: hidden; margin-bottom: -1px; }
.d6t .d1v-layer:first-child { border-radius: 8px 8px 0 0; }
.d6t .d1v-layer:last-child { border-radius: 0 0 8px 8px; margin-bottom: 0; }
.d6t .d1v-bar { width: 18px; flex-shrink: 0; }
.d6t .d1v-bar--you { background: #1B5162; }
.d6t .d1v-bar--plat { background: #618794; }
.d6t .d1v-lbody { flex: 1; display: flex; align-items: center; padding: 16px 20px; gap: 16px; }
.d6t .d1v-lbody--you { background: rgba(27,81,98,0.06); }
.d6t .d1v-lbody--plat { background: rgba(97,135,148,0.06); }
.d6t .d1v-lnum { width: 36px; height: 36px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 16pt; font-weight: 800; color: #fff; flex-shrink: 0; }
.d6t .d1v-lnum--you { background: #1B5162; }
.d6t .d1v-lnum--plat { background: #618794; }
.d6t .d1v-ltxt { flex: 1; }
.d6t .d1v-lt { font-size: 16pt; font-weight: 700; color: #1B3139; }
.d6t .d1v-ld { font-size: 14pt; color: #618794; margin-top: 2px; }
.d6t .d1v-ld code { background: #EEEDE9; padding: 1px 5px; border-radius: 3px; font-size: 14pt; }
.d6t .d1v-ltag { font-size: 14pt; font-weight: 600; padding: 4px 12px; border-radius: 4px; flex-shrink: 0; white-space: nowrap; }
.d6t .d1v-ltag--you { background: rgba(27,81,98,0.1); color: #1B5162; }
.d6t .d1v-ltag--plat { background: rgba(97,135,148,0.1); color: #618794; }
.d6t .d1v-footer { display: flex; gap: 16px; justify-content: center; padding: 14px 24px; background: #F9F7F4; border-top: 2px solid #E8E3DC; flex-wrap: wrap; }
.d6t .d1v-ftag { font-size: 14pt; font-weight: 600; color: #618794; padding: 4px 14px; border: 1.5px solid #DCE0E2; border-radius: 20px; background: #fff; }
/* D2 resource bindings table styles */
.d6t .d6t-tbl { width: 100%; border-spacing: 0; }
.d6t .d6t-tag { display: inline-block; padding: 3px 10px; border-radius: 4px; font-size: 14pt; font-weight: 600; margin: 2px 4px 2px 0; white-space: nowrap; }
.d6t .d6t-tag--hi { background: rgba(27,81,98,0.1); color: #1B5162; border: 1.5px solid #1B5162; }
.d6t .d6t-tag--lo { background: #fff; color: #618794; border: 1.5px solid #DCE0E2; }
</style>

<div class="d6t">
<input type="radio" name="d6tgrp" id="d6t1" checked>
<input type="radio" name="d6tgrp" id="d6t2">
<div class="d6t-tabs">
  <label class="d6t-tab" for="d6t1">App Architecture</label>
  <label class="d6t-tab" for="d6t2">Resource Bindings</label>
</div>
<div class="d6t-panels">
  <!-- Tab 1: App Architecture (from D1) -->
  <div class="d6t-panel d6t-p1">
    <div class="d1v-wrap">
      <div class="d1v-legend">
        <div class="d1v-leg"><div class="d1v-dot" style="background:#1B5162;"></div><span style="color:#1B5162;font-size:15pt;">You Write</span></div>
        <div class="d1v-leg"><div class="d1v-dot" style="background:#618794;"></div><span style="color:#618794;font-size:15pt;">Platform Manages</span></div>
      </div>
      <div class="d1v-stack">
        <div class="d1v-layer">
          <div class="d1v-bar d1v-bar--you"></div>
          <div class="d1v-lbody d1v-lbody--you">
            <div class="d1v-lnum d1v-lnum--you">1</div>
            <div class="d1v-ltxt">
              <div class="d1v-lt">Your Code</div>
              <div class="d1v-ld"><code style="font-size: 14pt;">server.py</code> + <code style="font-size: 14pt;">agent.py</code>. Same files from your notebook</div>
            </div>
            <div class="d1v-ltag d1v-ltag--you">You write</div>
          </div>
        </div>
        <div class="d1v-layer">
          <div class="d1v-bar d1v-bar--you"></div>
          <div class="d1v-lbody d1v-lbody--you">
            <div class="d1v-lnum d1v-lnum--you">2</div>
            <div class="d1v-ltxt">
              <div class="d1v-lt"><code style="font-size: 14pt;">app.yaml</code></div>
              <div class="d1v-ld">Startup command, env vars, resource bindings</div>
            </div>
            <div class="d1v-ltag d1v-ltag--you">You declare</div>
          </div>
        </div>
        <div class="d1v-layer">
          <div class="d1v-bar d1v-bar--plat"></div>
          <div class="d1v-lbody d1v-lbody--plat">
            <div class="d1v-lnum d1v-lnum--plat">3</div>
            <div class="d1v-ltxt">
              <div class="d1v-lt">Service Principal</div>
              <div class="d1v-ld">Auto-provisioned identity with scoped permissions</div>
            </div>
            <div class="d1v-ltag d1v-ltag--plat">Auto-created</div>
          </div>
        </div>
        <div class="d1v-layer">
          <div class="d1v-bar d1v-bar--plat"></div>
          <div class="d1v-lbody d1v-lbody--plat">
            <div class="d1v-lnum d1v-lnum--plat">4</div>
            <div class="d1v-ltxt">
              <div class="d1v-lt">Managed Runtime</div>
              <div class="d1v-ld">Python 3.11, HTTPS, auto-restart, dependency install</div>
            </div>
            <div class="d1v-ltag d1v-ltag--plat">Serverless</div>
          </div>
        </div>
      </div>
      <div class="d1v-footer">
        <div class="d1v-ftag">Same agent code from notebooks</div>
        <div class="d1v-ftag">Deploy via DABs or Apps UI</div>
        <div class="d1v-ftag">GA across AWS, Azure, GCP</div>
      </div>
    </div>
  </div>
  <!-- Tab 2: Resource Bindings (from D2) -->
  <div class="d6t-panel d6t-p2">
    <div style="padding:12px 24px;text-align:center;font-size:14pt;color:#618794;border-bottom:2px solid #E8E3DC;">
      Declared in <code style="font-size:14pt;background:#EEEDE9;padding:1px 6px;border-radius:4px;">app.yaml</code>. Scoped per app: 
      <span class="d6t-tag d6t-tag--hi" style="margin-left:6px;">Highlighted</span> = used in this lab
    </div>
    <table class="d6t-tbl">
      <tr>
        <th style="padding:12px 16px;font-size:14pt;font-weight:700;color:#1B5162;text-align:left;background:#F9F7F4;border-bottom:2px solid #E8E3DC;border-right:2px solid #E8E3DC;width:160px;">Data &amp; Storage</th>
        <td style="padding:12px 16px;font-size:14pt;color:#333;line-height:1.8;border-bottom:2px solid #E8E3DC;">
          <span class="d6t-tag d6t-tag--hi">Database</span>
          <span class="d6t-tag d6t-tag--lo">SQL Warehouse</span>
          <span class="d6t-tag d6t-tag--lo">UC Table</span>
          <span class="d6t-tag d6t-tag--lo">UC Volume</span>
          <span class="d6t-tag d6t-tag--lo">Vector Search Index</span>
          <div style="font-size:14pt;color:#618794;margin-top:6px;"><strong style="color:#1B5162;">Database</strong> = Lakebase Postgres for checkpoints + long-term memory</div>
        </td>
      </tr>
      <tr>
        <th style="padding:12px 16px;font-size:14pt;font-weight:700;color:#1B5162;text-align:left;background:#F9F7F4;border-bottom:2px solid #E8E3DC;border-right:2px solid #E8E3DC;width:160px;">AI &amp; ML</th>
        <td style="padding:12px 16px;font-size:14pt;color:#333;line-height:1.8;border-bottom:2px solid #E8E3DC;">
          <span class="d6t-tag d6t-tag--hi">Serving Endpoint</span>
          <span class="d6t-tag d6t-tag--hi">MLflow Experiment</span>
          <span class="d6t-tag d6t-tag--lo">Genie Space</span>
          <div style="font-size:14pt;color:#618794;margin-top:6px;"><strong style="color:#1B5162;">Serving Endpoint</strong> = LLM calls via ChatDatabricks &nbsp;|&nbsp; <strong style="color:#1B5162;">MLflow Experiment</strong> = traces for every invocation</div>
        </td>
      </tr>
      <tr>
        <th style="padding:12px 16px;font-size:14pt;font-weight:700;color:#1B5162;text-align:left;background:#F9F7F4;border-bottom:2px solid #E8E3DC;border-right:2px solid #E8E3DC;width:160px;">Governance</th>
        <td style="padding:12px 16px;font-size:14pt;color:#333;line-height:1.8;border-bottom:2px solid #E8E3DC;">
          <span class="d6t-tag d6t-tag--hi">UC Function</span>
          <span class="d6t-tag d6t-tag--lo">UC Connection</span>
          <span class="d6t-tag d6t-tag--lo">Secret</span>
          <div style="font-size:14pt;color:#618794;margin-top:6px;"><strong style="color:#1B5162;">UC Function</strong> = agent tools governed by Unity Catalog</div>
        </td>
      </tr>
      <tr>
        <th style="padding:12px 16px;font-size:14pt;font-weight:700;color:#1B5162;text-align:left;background:#F9F7F4;border-right:2px solid #E8E3DC;width:160px;">Platform</th>
        <td style="padding:12px 16px;font-size:14pt;color:#333;line-height:1.8;">
          <span class="d6t-tag d6t-tag--lo">Job</span>
          <span class="d6t-tag d6t-tag--lo">Databricks App</span>
          <div style="font-size:14pt;color:#618794;margin-top:6px;">Scheduled workflows and nested app references</div>
        </td>
      </tr>
    </table>
  </div>
</div>
</div>

<br/>
<style>
details summary span:first-child {
transition: transform 0.2s ease;
display: inline-block;
}
details[open] summary span:first-child {
transform: rotate(90deg);
}
</style>


## F. Deploy as a Databricks App

Now you will deploy the Bakehouse Sales Assistant as a **Databricks App** with a chat interface. The app uses **async** versions of CheckpointSaver and DatabricksStore for production performance.


### F1. Review the App Files

The lab setup created a `bakehouse-agent-app` folder in your workspace with all the files needed for deployment.



<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">App Files Overview</strong>
  <table style="width: 100%; border-collapse: collapse; margin-top: 8px; font-size: 14px;">
    <tr style="background-color: #d9e6f2;">
      <th style="text-align: left; padding: 8px; border: 1px solid #bbb;">File</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #bbb;">Purpose</th>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>agent.py</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Defines the LangGraph agent with async memory (AsyncCheckpointSaver, AsyncDatabricksStore)</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>server.py</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">FastAPI server that handles chat requests and streams responses</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>chat.html</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Frontend chat interface for franchise managers</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>app.yaml</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Databricks App configuration (resources, permissions, environment variables)</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>databricks.yml</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Databricks Asset Bundle manifest for deployment</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>pyproject.toml</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Python project metadata</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>requirements.txt</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Python dependencies for the app environment</td>
    </tr>
  </table>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Key Difference from Notebook Code</strong>
  <p>The app files use <strong>AsyncCheckpointSaver</strong> and <strong>AsyncDatabricksStore</strong> instead of their synchronous counterparts. This is strongly recommended for apps running on FastAPI to avoid blocking the async event loop. </p>
</div>

### F2. Verify app.yaml

The lab setup generated `app.yaml` with your catalog and Lakebase values filled in automatically. Run the cell below to see your values, then open the file to confirm they match.


In [0]:
print("=" * 60)
print("Verify these values in your app.yaml:")
print("=" * 60)
print(f"  APP_NAME:         {app_name}")
print(f"  CATALOG_NAME:     {catalog_name}")
print(f"  SCHEMA_NAME:      {schema_name}")
print(f"  LAKEBASE_PROJECT: {lakebase_autoscaling_project}")
print(f"  LAKEBASE_BRANCH:  {lakebase_autoscaling_branch}")
print("=" * 60)
print(f"\n  LLM Endpoint:     {LLM_ENDPOINT_NAME}")
print("  (Use this when adding the Serving Endpoint resource in F3)")


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Instructions</strong>
  <ol>
    <li>Open the <code>bakehouse-agent-app/app.yaml</code> file in your workspace.</li>
    <li>Confirm <code>CATALOG_NAME</code> and <code>LAKEBASE_AUTOSCALING_PROJECT</code> match the values printed above. These were filled in automatically by the lab setup.</li>
    <li>Note the <strong>LLM Endpoint</strong> name. You will need it when adding the Serving Endpoint resource in F3.</li>
  </ol>
</div>

### F3. Create the App and Add Resources

Select a card below to expand the setup instructions for that path. This section creates the app and binds its resources, but does **not** deploy the source code yet. You'll grant Lakebase permissions in F4, then deploy the app in F5.

<br></br>
<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">
<style>
.c2d-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }
.c2d-box { display: inline-block; width: 48%; margin-right: 4%; min-height: 90px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 12px 8px; text-align: center; box-sizing: border-box; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; }
.c2d-box:last-child { margin-right: 0; }
.c2d-box:hover { transform: translateY(-2px); }
.c2d-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.c2d-title { display: block; font-size: 13pt; font-weight: 700; color: #0B2026; line-height: 1.3; white-space: normal; pointer-events: none; }
.c2d-subtitle { display: block; font-size: 11pt; font-weight: 400; color: #666; line-height: 1.3; white-space: normal; pointer-events: none; margin-top: 4px; }
.c2d-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.c2d-detail-wrap.open { max-height: 1600px; opacity: 1; margin-top: 12px; }
.c2d-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; box-sizing: border-box; border-top: 7px solid #ccc; }
.c2d-steps-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 14px 14px; margin-bottom: 14px; font-size: 14.5pt; line-height: 1.6; }
.c2d-steps-block ol { margin: 8px 0 0 0; padding-left: 20px; }
.c2d-steps-block li { margin-bottom: 8px; }
.c2d-steps-block ul { margin: 4px 0 8px 20px; list-style: disc; }
.c2d-info-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 12px 14px; font-size: 14pt; line-height: 1.5; font-weight: 500; }
</style>
<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box;">
  <div class="c2d-row">
    <span class="c2d-box active" data-id="0" onclick="c2dSelect(0)" style="--pc:#1B5162; border-top-color:#1B5162;">
      <span class="c2d-title">Apps UI</span>
      <span class="c2d-subtitle">No Git folder</span>
    </span><span class="c2d-box" data-id="1" onclick="c2dSelect(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
      <span class="c2d-title">DABs UI</span>
      <span class="c2d-subtitle">Git folder</span>
    </span>
  </div>
  <div class="c2d-detail-wrap" id="c2d-detail-wrap">
    <div class="c2d-detail-card" id="c2d-detail-card"></div>
  </div>
</div>
<div id="c2d-content-0" style="display:none;">
  <p><strong>Deploy the Bundle (creates app + resource bindings)</strong></p>
  <ol>
    <li>Navigate inside the project folder and click on any file (e.g., <strong>databricks.yml</strong>). A rocket icon will appear. Click it.</li>
    <li>Click <strong>Deploy</strong> in the <strong>Deployments</strong> section. This DAB is configured to deploy to <strong>dev</strong> only.</li>
    <li>Once validation clears and the deployment summary appears, click <strong>Deploy</strong> again to confirm.</li>
    <li>Under <strong>Bundle resource</strong>, click the app link (e.g., <strong>agent-userid-dev</strong>) to open the <strong>Apps</strong> page. The <strong>Deployment output</strong> console will confirm a successful bundle deployment.</li>
  </ol>
  <p style="margin-top:14px;"><strong>Next:</strong> The app now exists and its resource bindings are in place, but the source code is not pushed yet. Continue to <strong>F4</strong> to grant Lakebase permissions before completing the deploy in F5.</p>
</div>
<div id="c2d-info-0" style="display:none;"><strong>Requires a Git folder.</strong> The DABs UI automates resource creation and permission grants from your <code>databricks.yml</code> file.</div>
<div id="c2d-content-1" style="display:none;">
  <p><strong>Step 1: Create App</strong></p>
  <ol>
    <li>In the upper right, click the <strong>Waffle icon</strong> and select <strong>Databricks Apps</strong>.</li>
    <li>Click <strong>Create App</strong>.</li>
    <li>Select <strong>Create a custom app</strong>.</li>
    <li>For <strong>App Name</strong>, enter the app name printed by the F2 cell above (e.g., <code>bakehouse-15363263-1780942446</code>). The name is derived from your username and capped at 29 characters to satisfy the Databricks Apps 30-character limit.</li>
    <li>Click <strong>Next: Configure Git</strong>.</li>
    <li>On the <strong>Configure Git repository</strong> screen, leave everything blank and click <strong>Next: Configure</strong>.</li>
    <li>Review but leave as it.  Click <strong>Create app</strong>.</li>
    <li>While waiting for the compute to provision proceed to Step 2.</li>
  </ol>
  <p><strong>Step 2: Add App Resources</strong></p>
  <p>The <strong>Resource Key</strong> is how <code>app.yaml</code> references each resource. If the key doesn't match, the app won't resolve its environment variables.</p>
  <ol>
  <li> Click on the three vertical dots in the upper right </li>
  <li> Click Edit, scroll to Resources </li>
  <li> Select <strong>+ Add Resource</strong> and add the resources below one at a time.
  </ol>
  <table style="width: 100%; border-collapse: collapse; margin-top: 8px; font-size: 14px;">
    <tr style="background-color: #d9e6f2;">
      <th style="text-align: left; padding: 8px; border: 1px solid #bbb;">Add Resource</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #bbb;">Search / Select</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #bbb;">Permission</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #bbb;">Resource Key</th>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;">Serving Endpoint</td>
      <td style="padding: 8px; border: 1px solid #ddd;">Search for your LLM endpoint (printed in F2)</td>
      <td style="padding: 8px; border: 1px solid #ddd;">Can Query</td>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>serving-endpoint</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;">MLflow Experiment</td>
      <td style="padding: 8px; border: 1px solid #ddd;">Click <strong>Create new experiment</strong>, name it <code>bakehouse_app</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Can Edit</td>
      <td style="padding: 8px; border: 1px solid #ddd;"><code>experiment</code></td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;">UC Function</td>
      <td style="padding: 8px; border: 1px solid #ddd;">Search for <code>top_products_by_franchise</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Can Execute</td>
      <td style="padding: 8px; border: 1px solid #ddd;">(default is fine)</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;">UC Function</td>
      <td style="padding: 8px; border: 1px solid #ddd;">Search for <code>franchise_performance</code></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Can Execute</td>
      <td style="padding: 8px; border: 1px solid #ddd;">(default is fine)</td>
    </tr>
    <tr>
      <td style="padding: 8px; border: 1px solid #ddd;">Database</td>
      <td style="padding: 8px; border: 1px solid #ddd;">Select your Lakebase instance, branch <strong>production</strong>, database <strong>databricks_postgres</strong></td>
      <td style="padding: 8px; border: 1px solid #ddd;">Can connect and create</td>
      <td style="padding: 8px; border: 1px solid #ddd;">(default is fine)</td>
    </tr>
  </table>
  <ol start="4">
    <li>Click <strong>Save</strong>.</li>
    <li>Navigate back to the app. Compute will continue provisioning in the background while you complete F4.</li>
  </ol>
  <p style="margin-top:14px;"><strong>Next:</strong> Continue to <strong>F4</strong> to grant Lakebase permissions before deploying the source code in F5.</p>
</div>
<div id="c2d-info-1" style="display:none;"><strong>No Git folder required.</strong> You configure app resources manually on the Overview page.</div>
</div>
<script>
var C2D_CARDS = [
  { title: "Apps UI (no Git folder)", color: "#1B5162", bg: "rgba(27,81,98,0.10)", infoBg: "rgba(27,81,98,0.10)", content: document.getElementById("c2d-content-1").innerHTML, info: document.getElementById("c2d-info-1").innerHTML },
  { title: "DABs UI (Git folder)", color: "#FF5F46", bg: "rgba(255,95,70,0.10)", infoBg: "rgba(255,95,70,0.10)", content: document.getElementById("c2d-content-0").innerHTML, info: document.getElementById("c2d-info-0").innerHTML }
];
var c2dCurrent = null;
function c2dSelect(id) {
  var wrap = document.getElementById("c2d-detail-wrap");
  var card = document.getElementById("c2d-detail-card");
  var c = C2D_CARDS[id];
  document.querySelectorAll(".c2d-box").forEach(function(b) {
    b.classList.toggle("active", parseInt(b.dataset.id, 10) === id);
  });
  if (c2dCurrent === id) {
    wrap.classList.remove("open");
    document.querySelectorAll(".c2d-box").forEach(function(b) { b.classList.remove("active"); });
    c2dCurrent = null;
    return;
  }
  c2dCurrent = id;
  card.style.borderTopColor = c.color;
  card.innerHTML = "<div style=\"font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;\">" + c.title + "</div>" + "<div class=\"c2d-steps-block\" style=\"background:" + c.bg + ";border-color:" + c.color + ";\">" + c.content + "</div>" + "<div class=\"c2d-info-block\" style=\"background:" + c.infoBg + ";border-color:" + c.color + ";\">" + c.info + "</div>";
  wrap.classList.add("open");
}
c2dSelect(0);
</script>

### F4. Configure Lakebase Roles

The app's service principal needs access to your Lakebase instance to read and write memory data.




<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Grant Lakebase Permissions</strong>
  <p>Adding the Database resource in the previous step automatically created a role for the app&#39;s service principal and assigned it to your Lakebase instance. However, the role does not yet have permissions. You need to grant it <code>databricks_superuser</code>.</p>
  <ol>
    <li>In the upper right, click the <strong>Waffle icon</strong> and select <strong>Lakebase Postgres</strong> &rarr; <strong>Autoscaling</strong>.</li>
    <li>Select your Lakebase instance.</li>
    <li>Click <strong>Branches</strong> &rarr; <strong>production</strong>.</li>
    <li>Click the <strong>Roles &amp; Database</strong> tab.</li>
    <li>Find the role for your app&#39;s service principal (it has the same name as your app).</li>
    <li>Click on the kebab menu, select <strong>Edit role</strong>, and grant the role <strong><code>databricks_superuser</code></strong>.</li>
  </ol>
  <p>Without this step, the app will fail to connect to Lakebase for memory operations.</p>
</div>

<div style="border-left: 4px solid #ff9800; background: #fff8e1; padding: 12px 16px; margin: 16px 0; border-radius: 4px;">
  <strong style="color: #e65100;">&#9888; Lab Only</strong>
  <p>The <code>databricks_superuser</code> role is used here for lab convenience. In production, create a custom role with the minimum required permissions for your application.</p>
</div>

Once the role is granted, continue to **F5** to deploy the app's source code.

### F5. Push the Source Code and Start the App

Lakebase permissions are now in place. Select the same path you chose in F3 to complete the deploy.

<br></br>
<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif;">
<style>
.f5d-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }
.f5d-box { display: inline-block; width: 48%; margin-right: 4%; min-height: 90px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 12px 8px; text-align: center; box-sizing: border-box; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; }
.f5d-box:last-child { margin-right: 0; }
.f5d-box:hover { transform: translateY(-2px); }
.f5d-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.f5d-title { display: block; font-size: 13pt; font-weight: 700; color: #0B2026; line-height: 1.3; white-space: normal; pointer-events: none; }
.f5d-subtitle { display: block; font-size: 11pt; font-weight: 400; color: #666; line-height: 1.3; white-space: normal; pointer-events: none; margin-top: 4px; }
.f5d-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.f5d-detail-wrap.open { max-height: 2400px; opacity: 1; margin-top: 12px; }
.f5d-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; box-sizing: border-box; border-top: 7px solid #ccc; }
.f5d-steps-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 14px 14px; margin-bottom: 14px; font-size: 14.5pt; line-height: 1.6; }
.f5d-steps-block ol { margin: 8px 0 0 0; padding-left: 20px; }
.f5d-steps-block li { margin-bottom: 8px; }
.f5d-steps-block ul { margin: 4px 0 8px 20px; list-style: disc; }
.f5d-info-block { border-left: 4px solid; border-radius: 0 6px 6px 0; padding: 12px 14px; font-size: 14pt; line-height: 1.5; font-weight: 500; }
</style>
<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box;">
  <div class="f5d-row">
    <span class="f5d-box active" data-id="0" onclick="f5dSelect(0)" style="--pc:#1B5162; border-top-color:#1B5162;">
      <span class="f5d-title">Apps UI</span>
      <span class="f5d-subtitle">No Git folder</span>
    </span><span class="f5d-box" data-id="1" onclick="f5dSelect(1)" style="--pc:#FF5F46; border-top-color:#FF5F46;">
      <span class="f5d-title">DABs UI</span>
      <span class="f5d-subtitle">Git folder</span>
    </span>
  </div>
  <div class="f5d-detail-wrap" id="f5d-detail-wrap">
    <div class="f5d-detail-card" id="f5d-detail-card"></div>
  </div>
</div>
<div id="f5d-content-0" style="display:none;">
  <ol>
    <li>If you navigated away, return to your app: click the <strong>Waffle icon</strong> in the upper right and select <strong>Databricks Apps</strong>, then click your app.</li>
    <li>Wait for compute to finish provisioning. Once ready, the <strong>Deploy</strong> button in the top right will turn blue.</li>
    <li>Click <strong>Deploy</strong>.</li>
    <li>Navigate to the <strong>bakehouse-agent-app</strong> folder in your workspace, click the three vertical dots, select <strong>Copy URL/path</strong>, and choose <strong>Full path</strong>.</li>
    <li>Paste the full path into the <strong>Create deployment</strong> text box.</li>
    <li>Click <strong>Deploy</strong>. After a few minutes, your app will show a <strong>Running</strong> status.</li>
    <li>Click the endpoint URL to open the app.</li>
  </ol>
</div>
<div id="f5d-info-0" style="display:none;"><strong>No Git folder required.</strong> The Deploy button pushes the source code from your workspace path.</div>
<div id="f5d-content-1" style="display:none;">
  <ol>
    <li>If you navigated away, return to your app: click the <strong>Waffle icon</strong> in the upper right and select <strong>Databricks Apps</strong>, then click your app (e.g., <code>agent-userid-dev</code>).</li>
    <li>Click <strong>Start</strong> to provision the app's compute.</li>
    <li>Once the app is started, click <strong>Deploy</strong> to push the source code:
      <ul>
        <li>Navigate to the <strong>simple-agent</strong> source folder, click the three vertical dots next to the folder name, select <strong>Copy URL/path</strong> and choose <strong>Full path</strong>.</li>
        <li>Paste this in the <strong>source code path</strong> textbox in the app's <strong>Create deployment</strong> menu.</li>
        <li>Click <strong>Deploy</strong>.</li>
      </ul>
    </li>
    <li>Once the app shows <strong>Running</strong> status, click the endpoint to open the chat UI and verify the agent responds by sending a query like <em>"Please tell me about Apache Spark."</em></li>
  </ol>
</div>
<div id="f5d-info-1" style="display:none;"><strong>Requires a Git folder.</strong> The Start button provisions compute; Deploy pushes the source code from your workspace path.</div>
</div>
<script>
var F5D_CARDS = [
  { title: "Apps UI (no Git folder)", color: "#1B5162", bg: "rgba(27,81,98,0.10)", infoBg: "rgba(27,81,98,0.10)", content: document.getElementById("f5d-content-0").innerHTML, info: document.getElementById("f5d-info-0").innerHTML },
  { title: "DABs UI (Git folder)", color: "#FF5F46", bg: "rgba(255,95,70,0.10)", infoBg: "rgba(255,95,70,0.10)", content: document.getElementById("f5d-content-1").innerHTML, info: document.getElementById("f5d-info-1").innerHTML }
];
var f5dCurrent = null;
function f5dSelect(id) {
  var wrap = document.getElementById("f5d-detail-wrap");
  var card = document.getElementById("f5d-detail-card");
  var c = F5D_CARDS[id];
  document.querySelectorAll(".f5d-box").forEach(function(b) {
    b.classList.toggle("active", parseInt(b.dataset.id, 10) === id);
  });
  if (f5dCurrent === id) {
    wrap.classList.remove("open");
    document.querySelectorAll(".f5d-box").forEach(function(b) { b.classList.remove("active"); });
    f5dCurrent = null;
    return;
  }
  f5dCurrent = id;
  card.style.borderTopColor = c.color;
  card.innerHTML = "<div style=\"font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;\">" + c.title + "</div>" + "<div class=\"f5d-steps-block\" style=\"background:" + c.bg + ";border-color:" + c.color + ";\">" + c.content + "</div>" + "<div class=\"f5d-info-block\" style=\"background:" + c.infoBg + ";border-color:" + c.color + ";\">" + c.info + "</div>";
  wrap.classList.add("open");
}
f5dSelect(0);
</script>

### F6. Test the Deployed App

Once the app reaches **Running** status, open it in your browser and test both memory types.



<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Test Plan</strong>
  <p><strong>Test 1: Short-Term Memory</strong></p>
  <ol>
    <li>Open the app URL.</li>
    <li>Send:</li>
  </ol>
  <div style="position: relative; max-width: 900px; margin: 8px 0 16px 0;">
    <div id="msg1" style="background: #1b3139; border-radius: 8px; padding: 18px 22px; color: #e0e0e0; font-family: monospace; font-size: 12pt; line-height: 1.6; border-left: 5px solid #02A36F;">Hi, I'm Yuki and I manage the Tokyo Treats franchise in Shibuya. What are our top products?</div>
    <button onclick="var t=document.getElementById('msg1').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:10px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <ol start="3">
    <li>Follow up:</li>
  </ol>
  <div style="position: relative; max-width: 900px; margin: 8px 0 16px 0;">
    <div id="msg2" style="background: #1b3139; border-radius: 8px; padding: 18px 22px; color: #e0e0e0; font-family: monospace; font-size: 12pt; line-height: 1.6; border-left: 5px solid #2574B5;">How does our revenue compare to other franchises in Japan?</div>
    <button onclick="var t=document.getElementById('msg2').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:10px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <ol start="4">
    <li>Verify the agent remembers your franchise across turns.</li>
  </ol>
  <p><strong>Test 2: Long-Term Memory</strong></p>
  <ol>
    <li><strong>Refresh the browser</strong> (this clears the in-page conversation history array, effectively starting a new thread; the <code>user_id</code> is preserved by the app's authentication system).</li>
    <li>Send:</li>
  </ol>
  <div style="position: relative; max-width: 900px; margin: 8px 0 16px 0;">
    <div id="msg3" style="background: #1b3139; border-radius: 8px; padding: 18px 22px; color: #e0e0e0; font-family: monospace; font-size: 12pt; line-height: 1.6; border-left: 5px solid #8B5CF6;">What do you remember about me?</div>
    <button onclick="var t=document.getElementById('msg3').textContent,a=document.createElement('textarea');a.value=t;a.style.position='fixed';a.style.left='-9999px';document.body.appendChild(a);a.select();document.execCommand('copy');document.body.removeChild(a);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:10px;background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.25);color:#e0e0e0;border-radius:4px;padding:4px 10px;font-size:11px;cursor:pointer;">Copy</button>
  </div>
  <ol start="3">
    <li>Verify the agent greets you by name and references your franchise.</li>
  </ol>
  <p><strong>Test 3: MLflow Traces</strong></p>
  <ol>
    <li>Open the MLflow experiment linked to your app.</li>
    <li>Inspect the traces to see <code>recall_memories</code>, <code>save_memory</code>, and UC function tool calls.</li>
  </ol>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#0d47a1;">Why does the agent call recall_memories on every turn?</strong>
  <p>You may notice that the agent calls <code>recall_memories</code> on follow-up messages within the same session, not just the first message. This is expected. The system prompt instructs the agent to "call recall_memories at the START of every conversation," but the LLM cannot distinguish the first turn from subsequent turns in the same thread. Short-term memory IS working (the agent references prior turns), but the recall instruction fires regardless.</p>
  <p>In production, you could refine the system prompt to say "only recall if you have not already done so in this conversation," or add logic in the agent code to skip recall after the first turn. For this lab, the behavior is harmless and confirms the memory tools are active.</p>
</div>

## G. Explore Both Memory Systems in Lakebase

Both short-term memory (CheckpointSaver) and long-term memory (DatabricksStore) persist their data in the same Lakebase instance. This section explores what each system stored and how they work together.


### G1. Short-Term Memory: All Conversation Threads

Every conversation thread from the notebook **and** the deployed app is stored in the same Lakebase `checkpoint_writes` table. Let's discover all of them.


In [0]:
import psycopg
from databricks.sdk import WorkspaceClient
from databricks_langchain import CheckpointSaver

# Connect directly to Lakebase to discover all threads
w_client = WorkspaceClient()
ep_name = f"projects/{lakebase_autoscaling_project}/branches/{lakebase_autoscaling_branch}/endpoints/primary"
endpoint = w_client.postgres.get_endpoint(name=ep_name)
cred = w_client.postgres.generate_database_credential(endpoint=ep_name)
pg_conn_str = f"host={endpoint.status.hosts.host} dbname=databricks_postgres user={w_client.current_user.me().user_name} password={cred.token} sslmode=require"

known_threads = {
    session_thread: "D1: Short-term (4 turns)",
    stateless_thread: "D2: Stateless",
    new_thread: "D3: Test recall",
    update_thread: "D3: Update preference",
}

# Discover ALL thread IDs from checkpoint_writes
with psycopg.connect(pg_conn_str) as pg_conn:
    all_thread_ids = {r[0] for r in pg_conn.execute("SELECT DISTINCT thread_id FROM checkpoint_writes").fetchall()}

# Label: known notebook threads vs app conversations
threads = {}
app_counter = 1
for tid in sorted(all_thread_ids):
    if tid in known_threads:
        threads[tid] = known_threads[tid]
    else:
        threads[tid] = f"App conversation {app_counter} (Section F)"
        app_counter += 1

with CheckpointSaver(project=lakebase_autoscaling_project, branch=lakebase_autoscaling_branch) as saver:
    print(f"{'Session':<35} {'Thread ID':<40} {'Checkpoints':<12} {'Messages'}")
    print("-" * 100)
    for tid, label in threads.items():
        cps = list(saver.list({"configurable": {"thread_id": tid}}))
        msg_count = len(cps[0].checkpoint.get("channel_values", {}).get("messages", [])) if cps else 0
        print(f"{label:<35} {tid[:36]+'...':<40} {len(cps):<12} {msg_count}")


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul>
    <li><strong>D1 (4 turns)</strong> has the most checkpoints and messages, as the full multi-turn conversation is preserved.</li>
    <li><strong>D2 (stateless)</strong> has very few messages, as it is a fresh thread with no history.</li>
    <li><strong>D3 threads (test recall, update preference)</strong> each have fewer messages since they are single-turn new threads, but the agent still knew Jordan via long-term memory.</li>
    <li><strong>App conversations</strong> from Section F appear alongside the notebook threads, because the deployed app and the notebook share the same Lakebase checkpoint storage.</li>
  </ul>
</div>

### G2. Long-Term Memory: All Manager Namespaces

Every manager identity from the notebook **and** the deployed app is stored in the same Lakebase `store` table. Each namespace contains the curated facts the agent saved about that manager.


In [0]:
all_namespaces = list(store.list_namespaces(prefix=("managers",)))

for ns in all_namespaces:
    ns_id = ns[-1] if ns else "unknown"
    memories = list(store.search(ns, query="all", limit=20))

    if ns_id == manager_user_id:
        label = "← notebook (Section D)"
    elif ns_id not in (manager_user_id,) and not ns_id.startswith("anonymous"):
        label = "← deployed app (Section F)"
    else:
        label = ""

    print(f"\nNamespace: {ns_id}  {label}")
    print(f"Facts stored: {len(memories)}")
    for item in memories:
        print(f"  [{item.key}]: {item.value['content']}")

if not all_namespaces:
    print("No long-term memories found.")


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul>
    <li><strong>manager_jordan_sf</strong> has curated facts (name, franchise, preferred product now Golden Gate Ginger) that survived across all D3 sessions, each with a completely new thread.</li>
    <li><strong>Your Databricks username</strong> may appear as a separate namespace if you tested the deployed app in Section F, confirming that the notebook and the production app share the same Lakebase store.</li>
  </ul>
  <p><strong>Summary:</strong> Short-term memory (checkpoints) handles context <em>within</em> a thread. Long-term memory (store) handles knowledge <em>across</em> threads. Together, they support both single-session and returning-user scenarios.</p>
</div>

### G3. Inside a Checkpoint: What the Agent Sees

Let's look at the actual conversation messages stored in an **app conversation** checkpoint. This is exactly what the LLM receives as context when the next turn arrives.


In [0]:
# Pick the first app thread (if any), otherwise fall back to the D3 test recall thread
app_thread_ids = [tid for tid in threads if threads[tid].startswith("App conversation")]
inspect_thread = app_thread_ids[0] if app_thread_ids else new_thread
inspect_label = threads.get(inspect_thread, "D3: Test recall")

with CheckpointSaver(
    project=lakebase_autoscaling_project,
    branch=lakebase_autoscaling_branch,
) as saver:
    config = {"configurable": {"thread_id": inspect_thread}}
    checkpoints = list(saver.list(config, limit=1))

    if checkpoints:
        latest = checkpoints[0]
        messages = latest.checkpoint.get("channel_values", {}).get("messages", [])

        print(f"Thread: {inspect_label}  ({inspect_thread[:36]}...)")
        print(f"Latest checkpoint: {len(messages)} messages stored:\n")
        print("=" * 80)

        for i, msg in enumerate(messages):
            role = getattr(msg, "type", "unknown")
            content = getattr(msg, "content", "")

            if isinstance(content, str) and len(content) > 200:
                display_content = content[:200] + "... [truncated]"
            elif isinstance(content, list):
                display_content = str(content)[:200] + "... [truncated]" if len(str(content)) > 200 else str(content)
            else:
                display_content = content

            print(f"\n[{i+1}] {role.upper()}")
            print(f"    {display_content}")
            print("-" * 80)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul style="margin-bottom: 0;">
    <li>Each message has a <strong>role</strong>: <code>human</code> (user queries), <code>ai</code> (agent responses), and <code>tool</code> (UC function results and memory operations).</li>
    <li>Look for <strong>tool messages</strong> showing <code>save_memory</code> confirmations and data tool returns. These are part of the checkpoint, not just the user-visible conversation.</li>
    <li>The <strong>last message</strong> is the agent's most recent response. Everything before it is the context the LLM used to generate that response. This is short-term memory in its raw form.</li>
    <li>This is the <strong>same data</strong> whether the conversation happened in the notebook or the deployed app, as both write to the same Lakebase tables.</li>
  </ul>
</div>

### G4. Inside the Test Recall Thread: Where Long-Term Memory Fills the Gap

The Session 2 "Test Recall" started a **brand-new thread**: the checkpoint was empty. Yet the agent knew Jordan's name, franchise, and signature product. Let's look at the raw messages to see exactly how `recall_memories` bridged the gap.


In [0]:
with CheckpointSaver(
    project=lakebase_autoscaling_project,
    branch=lakebase_autoscaling_branch,
) as saver:
    config = {"configurable": {"thread_id": new_thread}}
    checkpoints = list(saver.list(config, limit=1))

    if checkpoints:
        latest = checkpoints[0]
        messages = latest.checkpoint.get("channel_values", {}).get("messages", [])

        print(f"Thread: D3 Test Recall  ({new_thread[:36]}...)")
        print(f"Latest checkpoint: {len(messages)} messages stored:\n")
        print("=" * 80)

        for i, msg in enumerate(messages):
            role = getattr(msg, "type", "unknown")
            content = getattr(msg, "content", "")

            if isinstance(content, str) and len(content) > 200:
                display_content = content[:200] + "... [truncated]"
            elif isinstance(content, list):
                display_content = str(content)[:200] + "... [truncated]" if len(str(content)) > 200 else str(content)
            else:
                display_content = content

            print(f"\n[{i+1}] {role.upper()}")
            print(f"    {display_content}")
            print("-" * 80)


<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#2e7d32;">&#10003; What to Observe</strong>
  <ul style="margin-bottom: 0;">
    <li>The <strong>first message</strong> is the user's query, with no prior conversation context. The checkpoint started empty.</li>
    <li>Look for the <strong>tool message from <code>recall_memories</code></strong>: it returns the stored facts (name, franchise, preferred product) that were saved during earlier sessions.</li>
    <li>After recall, the agent has enough context to <strong>greet Jordan by name</strong> and query Golden Crumbs data, all without being told again.</li>
    <li>Compare this to G3: that thread had a full conversation history in the checkpoint. This thread had <strong>nothing</strong>: long-term memory was the only source of context.</li>
  </ul>
</div>

## H. Cleanup

When you are done exploring, clean up the resources created during this lab. This includes the Databricks App, MLflow experiment, Lakebase role assignment, and the app folder in your workspace.



<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; margin: 16px 0; border-radius: 4px;">
  <strong style="color:#c62828;">&#9888; Cleanup Steps</strong>
  <ol>
    <li><strong>Stop and delete the Databricks App:</strong> Click the <strong>Waffle icon</strong> &rarr; <strong>Databricks Apps</strong>, select your app, click <strong>Stop</strong>, then <strong>Delete</strong>.</li>
    <li><strong>Delete the MLflow Experiment:</strong> Open the experiment in the MLflow UI and delete it.</li>
    <li><strong>Remove Lakebase Role:</strong> Click the <strong>Waffle icon</strong> &rarr; <strong>Lakebase Postgres</strong> &rarr; <strong>Autoscaling</strong>, open your instance, go to <strong>Roles</strong>, and revoke the app's service principal.</li>
    <li><strong>Delete the app folder:</strong> Remove the <code>bakehouse-agent-app</code> folder from your workspace.</li>
  </ol>
  <p>The UC functions and Lakebase instance can be kept for further experimentation or cleaned up by running the classroom cleanup script.</p>
</div>

## Conclusion

You have built a complete **stateful AI assistant** from the ground up:


1. **UC Function Tools**: Created SQL functions that query real bakehouse sales data and wrapped them as LangChain tools
1. **Long-Term Memory Tools**: Implemented `save_memory` and `recall_memories` using DatabricksStore backed by Lakebase
1. **LangGraph Agent**: Built a multi-tool agent with a system prompt that proactively manages memory
1. **Short-Term Memory**: Tested multi-turn conversations within a session using CheckpointSaver
1. **Cross-Session Memory**: Verified that facts persist across completely new conversation threads
1. **Production Deployment**: Deployed the assistant as a Databricks App with async memory and a chat interface

This architecture demonstrates patterns that can be applied in production AI assistants. The combination of short-term context (checkpoints) and long-term knowledge (store) supports agents that can maintain continuity and personalization across interactions.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>